# Análisis de datos de la Dark Web (ingesta realizada del 10 de diciembre del 2025 al 16 de junio de 2026)

<div class="seccion-ocultar">

## Preparación del entorno

</div>

<div class="seccion-ocultar">

### Imports:
Estos son todos los imports necesarios para la ejecución completa de todas las celdas del trabajo, separados por secciones dependiendo de la función que realicen.

</div>

In [ ]:
# --- 1. LIBRERÍAS DEL SISTEMA Y UTILIDADES ---
import os
import re
import time
import base64
import random
import itertools
from itertools import combinations
from datetime import datetime
from urllib.parse import urlparse
import subprocess
import sys

# --- 2. GESTIÓN DE BASES DE DATOS (Mongo y Neo4j) ---
from pymongo import MongoClient
from bson import ObjectId
import gridfs
from gridfs import GridFS
from neo4j import GraphDatabase, exceptions as neo4j_exceptions

# --- 3. PROCESAMIENTO DE DATOS Y ESTADÍSTICA ---
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import chisquare, spearmanr, kendalltau, mannwhitneyu
from scipy.spatial import ConvexHull
from bs4 import BeautifulSoup
from tqdm import tqdm

# --- 4. MACHINE LEARNING Y MINERÍA DE TEXTO ---
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.preprocessing import StandardScaler, Normalizer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score, adjusted_rand_score 

# --- 5. ANÁLISIS DE REDES (Grafos) ---
import networkx as nx
import community.community_louvain as community_louvain
from pyvis.network import Network

# --- 6. VISUALIZACIÓN DE DATOS (Estática) ---
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from matplotlib.patches import Polygon
import seaborn as sns
from wordcloud import WordCloud
from matplotlib.patches import Patch

# --- 7. VISUALIZACIÓN INTERACTIVA (Plotly e ITables) ---
import plotly.graph_objects as go
import plotly.express as px
import itables
import itables.options as opt
from itables import show

# --- 8. INTERFAZ JUPYTER / IPYWIDGETS ---
from IPython.display import display, HTML, IFrame, clear_output, FileLink, Javascript
import warnings

# --- 9. MEJORA CALIDAD IMAGENES ---
%config InlineBackend.figure_format = 'svg'

<div class="seccion-ocultar">

### Conexiones:

</div>

<div class="seccion-ocultar">

Aquí vamos a configurar las diversas conexiones necesarias para el uso de las bases de datos utilizadas.

</div>

<div class="seccion-ocultar">

Primero de todo, se indican las variables globales necesarias en los códigos y conexiones.

</div>

In [ ]:
NEO_URI = "bolt://localhost:7688"
NEO_USER = "neo4j"
NEO_PASS = "test1234"
MONGO_URI = "mongodb://localhost:27017"
MONGO_DB_NAME = "darkweb_tfg"

<div class="seccion-ocultar">

Se crean unas variables globales para todos los drivers de cada base de datos a utilizar.

</div>

In [ ]:
neo_driver = None
mongo_client = None

<div class="seccion-ocultar">

Establece la conexión en las diversas bases de datos, tanto MongoDB como Neo4J

</div>

In [ ]:
def get_mongo_connection():
    global mongo_client
    if mongo_client is None:
        try:
            mongo_client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=5000)
            mongo_client.admin.command('ismaster')
            print("Conexión a MongoDB exitosa.")
        except Exception as e:
            print(f"Error al conectar a MongoDB: {e}")
            mongo_client = None
    return mongo_client

In [ ]:
def get_neo_driver():
    global neo_driver
    if neo_driver is None:
        try:
            neo_driver = GraphDatabase.driver(
                NEO_URI, 
                auth=(NEO_USER, NEO_PASS), 
                encrypted=False, 
                connection_timeout=5.0
            )
            neo_driver.verify_connectivity()
            print("Conexión a Neo4j exitosa.")
        except Exception as e:
            print(f"Error al conectar a Neo4j: {e}")
            neo_driver = None
    return neo_driver

<div class="seccion-ocultar">

Finalmente, se tienen las conexiones listas e iniciadas.

</div>

In [ ]:
#OCULTAR
get_mongo_connection()
get_neo_driver()

## Análisis de datos

### Cola de seeds (semillas) en MongoDB

Con este código se puede ver la distribución de la cola de los seeds que hay en MongoDB.  
Se puede ver en dos barras:
- **Pendientes:** que son las seeds que están esperando a ser procesadas.
- **Procesados:** que detro de los procesados tenemos 3 tipos distintos:
   - **Ingestados:** Ya están procesados y se tiene toda la información de cada uno de ellos.
   - **Datos insuficientes:** Se vio la página pero no cumple con el minimo datos en la página, entonces no se guarda ni coge la información (páginas únicamente de fotos, solo pagos...)
   - **Página inexistente:** Se vio la página pero la página no existe, es decir, tenemos la URL pero la página al intentar abrirla no existe porque ha desaparecido o la han borrado (es algo típico en la Dark Web)

In [ ]:
# SILENCIAR WARNINGS
warnings.filterwarnings('ignore')
opt.style = "margin-left: 0; margin-right: auto; width: auto;"

# CONEXIÓN
if mongo_client:
    db = mongo_client[MONGO_DB_NAME]
    seeds_collection = db['seeds']
    
    # OBTENCIÓN DE DATOS DESDE MONGO
    total_seeds = seeds_collection.count_documents({})
    processed_seeds = seeds_collection.count_documents({'status': 'ingested'})
    unprocessed_seeds = seeds_collection.count_documents({'status': 'pending'})
    short_content = seeds_collection.count_documents({'status': 'discarded', 'discard_reason': 'too_short_content'})
    max_attempts_failed = seeds_collection.count_documents({'status': 'failed_perm'})

    # PREPARACIÓN DE LA GRÁFICA
    plt.figure(figsize=(14, 4), facecolor='none') 
    ax = plt.gca()
    ax.set_facecolor('none')
    
    # PENDIENTES
    plt.barh('Pendientes', unprocessed_seeds, color='#F6AD55', edgecolor='black', height=0.4)
    plt.text(unprocessed_seeds * 1.02, 0, f"{unprocessed_seeds} ({unprocessed_seeds/total_seeds*100:.1f}%)", 
             va='center', fontweight='bold', fontsize=10)
    
    # PROCESADOS
    plt.barh('Procesados', processed_seeds, color='#38A169', label='Ingested', height=0.4, edgecolor='black')
    plt.barh('Procesados', short_content, left=processed_seeds, color='#F56565', label='Datos Insuficientes', height=0.4, edgecolor='black')
    plt.barh('Procesados', max_attempts_failed, left=processed_seeds + short_content, color='#C53030', label='Página Inexistente', height=0.4, edgecolor='black')

    total_intento = processed_seeds + short_content + max_attempts_failed
    plt.text(total_intento * 1.02, 1, f"Total: {total_intento} ({total_intento/total_seeds*100:.1f}%)", 
             va='center', fontweight='bold', fontsize=10)

    plt.title('Distribución de la Cola de Seeds', fontsize=15, fontweight='bold', pad=25, color='#333')
    plt.xlim(0, max(unprocessed_seeds, total_intento) * 1.4)
    
    plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.25), ncol=3, shadow=False, frameon=False, fontsize=9)
    
    plt.gca().invert_yaxis()
    plt.grid(axis='x', linestyle='--', alpha=0.3)
    sns.despine() 
    plt.tight_layout()
    plt.show()

    # TABLA
    print("\n📊 TABLA RESUMEN DE ESTADOS:")
    
    df_status = pd.DataFrame({
        'Estado / Motivo': ['Pendientes', 'Ingested', 'D. Insuficientes', 'P. Inexistente'],
        'Cantidad': [unprocessed_seeds, processed_seeds, short_content, max_attempts_failed],
        'Porcentaje (%)': [
            f"{(unprocessed_seeds/total_seeds)*100:.2f}%",
            f"{(processed_seeds/total_seeds)*100:.2f}%",
            f"{(short_content/total_seeds)*100:.2f}%",
            f"{(max_attempts_failed/total_seeds)*100:.2f}%"
        ]
    })
    
    display(HTML("""
        <style>
            .itables table { 
                margin-left: 0 !important; 
                margin-right: auto !important; 
                width: 700px !important; 
                max-width: 800px !important;
                font-size: 0.85em !important;
                border-collapse: collapse !important;
                table-layout: fixed !important;
            }
            .itables th, .itables td { 
                padding: 10px 15px !important; 
                white-space: nowrap !important;
                overflow: hidden;
                text-overflow: ellipsis;
            }
            .itables th { background-color: #f7fafc !important; border-bottom: 2px solid #edf2f7 !important; }
            .itables td { border-bottom: 1px solid #edf2f7 !important; }
        </style>
    """))
    
    show(df_status, 
         paging=False, 
         search=False, 
         info=False, 
         dom='t',
         autoWidth=False,
         columnDefs=[
             {"width": "300px", "targets": 0}, # Estado / Motivo
             {"width": "200px", "targets": 1}, # Cantidad
             {"width": "200px", "targets": 2}, # Porcentaje (%)
             {"className": "dt-left", "targets": "_all"}
         ])

    # GUARDADO
    ruta_graficas = 'Graficas'
    os.makedirs(ruta_graficas, exist_ok=True)
    path_archivo = os.path.join(ruta_graficas, 'estado_seeds.csv')
    df_status.to_csv(path_archivo, index=False)
    
    csv_data = df_status.to_csv(index=False)
    b64_csv = base64.b64encode(csv_data.encode()).decode()
    payload = f"data:text/csv;base64,{b64_csv}"
    
    display(HTML(f"""
    <div style="text-align: left; margin-top: 15px;">
        <span style="font-size: 10px; color: #718096; display: block; margin-bottom: 8px;">📁 Guardado en: {path_archivo}</span>
        <a href="{payload}" download="estado_seeds.csv" style="
            display: inline-block; background-color: #4299e1; color: white;
            padding: 6px 14px; text-decoration: none; border-radius: 6px;
            font-weight: 600; font-family: sans-serif; font-size: 12px;
        ">📥 Descargar CSV</a>
    </div>
    """))

<div class="seccion-ocultar">

### Colores utilizados en el proyecto

</div>

<div class="seccion-ocultar">


Paleta de colores a utilizar en el proyecto para poder tener un entorno con todos los colores separados por las diversas tematicas.

</div>

In [ ]:
colores_tfg = {
    'drugs':       '#48BB78', 'carding':     '#ECC94B',
    'hacking':     '#0BC5EA', 'ransomware':  '#F56565',
    'weapon':      '#DD6B20', 'marketplace': '#4299E1',
    'phone':       '#A0AEC0', 'fraud':       '#9F7AEA'
}

### Análisis porcentual sobre el total de datos

Sobre el total de la muestra de datos que se encuentra en Neo4J se puede ver el porcentaje que representa cada tematica sobre el total de datos ingestados.

In [ ]:
# SILENCIAR

opt.warn_on_undocumented_option = False
pd.options.mode.chained_assignment = None 

# CONEXION
if neo_driver:
    session = neo_driver.session()

    # CONSULTA: Obtenemos el conteo de páginas por cada término
    query_alcance = """
    MATCH (root:Term)
    OPTIONAL MATCH (synonym:Synonym)-[:IS_SYNONYM_OF]->(root)
    WITH root, collect(toLower(synonym.name)) + toLower(root.name) AS all_keywords
    OPTIONAL MATCH (p:Page)
    WHERE ANY(word IN all_keywords WHERE toLower(p.text) CONTAINS toLower(word))
    RETURN root.name AS term_name, COUNT(DISTINCT p) AS page_count
    """
    
    try:
        results = session.run(query_alcance).data()
        session.close()
        
        df_alcance = pd.DataFrame(results)

        # Cálculo de hegemonía relativa
        total_paginas_categorias = df_alcance['page_count'].sum()
        df_alcance['Alcance (%)'] = (df_alcance['page_count'] / total_paginas_categorias) * 100
        
        # Ordenamos y Top 10
        df_alcance = df_alcance.sort_values(by='Alcance (%)', ascending=False).head(10)
        
        # --- GRÁFICA ---
        plt.figure(figsize=(14, 6), facecolor='none')
        ax = plt.gca()
        ax.set_facecolor('none')
        
        # Colores dinámicos (asegúrate de tener definido colores_tfg)
        lista_colores = [colores_tfg.get(x.lower(), '#2D3748') for x in df_alcance['term_name']]

        plot = sns.barplot(
            data=df_alcance, 
            x='Alcance (%)', 
            y='term_name', 
            hue='term_name',
            palette=lista_colores, 
            legend=False,
            edgecolor='black', 
            linewidth=1.5,
            alpha=0.85
        )

        max_val = df_alcance['Alcance (%)'].max()
        for p in plot.patches:
            val = p.get_width()
            if val > 0:
                plt.text(val + (max_val * 0.01), p.get_y() + p.get_height() / 2, 
                         f'{val:.1f}%', va='center', fontweight='bold', fontsize=11, color='#2d3748')
        
        ax.set_yticks(range(len(df_alcance)))
        ax.set_yticklabels([str(n).upper() for n in df_alcance['term_name']], fontweight='bold', fontsize=11)
        for i, tick in enumerate(ax.get_yticklabels()):
            tick.set_color(lista_colores[i])
            
        plt.title('Distribución Relativa de la Ontología', 
                  fontsize=16, fontweight='bold', pad=25, color='#333')
        plt.xlabel('Porcentaje de representatividad sobre el total categorizado (%)', fontsize=10, fontweight='bold')
        plt.ylabel('', fontsize=0)
        plt.grid(axis='x', linestyle='--', alpha=0.3)
        plt.xlim(0, max_val * 1.15)
        sns.despine()
        plt.tight_layout()
        
        # Guardado de imagen
        os.makedirs('Graficas', exist_ok=True)
        path_img = 'Graficas/barras_distribucion.svg'
        plt.savefig(path_img, dpi=300, bbox_inches='tight', facecolor='white')
        plt.show()

        # --- TABLA (FORMATO UNIFICADO) ---
        df_tabla = df_alcance.copy().reset_index(drop=True)
        df_tabla['term_name'] = df_tabla['term_name'].str.upper()
        df_tabla['Alcance (%)'] = df_tabla['Alcance (%)'].round(2)
        df_tabla.columns = ['Categoría', 'Páginas Únicas', 'Alcance (%)']
        
        print("\n📊 DETALLE DE DISTRIBUCIÓN RELATIVA:")
        
        display(HTML("""
            <style>
                .jp-RenderedHTMLCommon table, .rendered_html table { margin-left: 0 !important; margin-right: auto !important; }
                .itables { text-align: left !important; margin-left: 0 !important; }
                .itables table { 
                    margin-left: 0 !important; margin-right: auto !important; 
                    width: 700px !important; max-width: 800px !important;
                    font-size: 0.85em !important; border-collapse: collapse !important;
                    table-layout: fixed !important; 
                }
                .itables th, .itables td { 
                    padding: 10px 15px !important; text-align: left !important;
                    white-space: nowrap !important; overflow: hidden; text-overflow: ellipsis;
                }
                .itables th { background-color: #f7fafc !important; border-bottom: 2px solid #edf2f7 !important; }
                .itables td { border-bottom: 1px solid #edf2f7 !important; }
            </style>
            """))

        show(df_tabla, 
             paging=False, search=False, info=False, dom='t', autoWidth=False,
             columnDefs=[
                 {"width": "300px", "targets": 0, "className": "dt-left"},
                 {"width": "200px", "targets": 1, "className": "dt-left"},
                 {"width": "200px", "targets": 2, "className": "dt-left"},
                 {"className": "dt-left", "targets": "_all"}
             ])

        # --- SECCIÓN DE GUARDADO Y BOTÓN CSV ---
        path_csv = 'Graficas/clasificacion_tematica.csv'
        df_tabla.to_csv(path_csv, index=False)
        
        csv_data = df_tabla.to_csv(index=False)
        b64_csv = base64.b64encode(csv_data.encode()).decode()
        payload = f"data:text/csv;base64,{b64_csv}"
        
        display(HTML(f"""
        <div style="text-align: left; margin-top: 15px;">
            <span style="font-size: 10px; color: #718096; display: block; margin-bottom: 8px;">📁 Guardado en: {path_csv}</span>
            <a href="{payload}" download="hegemonia_tematica.csv" style="
                display: inline-block; background-color: #4299e1; color: white;
                padding: 6px 14px; text-decoration: none; border-radius: 6px;
                font-weight: 600; font-family: sans-serif; font-size: 12px;
            ">📥 Descargar CSV</a>
        </div>
        """))

    except Exception as e:
        print(f"❌ Error: {e}")

### Comparativa entre páginas únicas y frecuencia de menciones

Ahora se realiza una gráfica comparando la frecuencía de aparición por páginas con la cantidad de menciones de cada uno de los términos raíz que tenemos.

In [ ]:
# SILENCIAR
import itables.options as opt
opt.warn_on_undocumented_option = False
pd.options.mode.chained_assignment = None 

# CONEXION
if neo_driver:
    session = neo_driver.session()

    # CONSULTA
    query_mentions = """
    MATCH (t:Term)
    OPTIONAL MATCH (t)<-[:IS_SYNONYM_OF]-(s:Synonym)
    OPTIONAL MATCH (s)<-[:MENTIONS]-(p)
    RETURN t.name AS term_name, count(p) AS total_mentions
    """
    query_pages = """
    MATCH (root:Term)
    OPTIONAL MATCH (synonym:Synonym)-[:IS_SYNONYM_OF]->(root)
    WITH root, collect(toLower(synonym.name)) + toLower(root.name) AS all_keywords
    OPTIONAL MATCH (p:Page)
    WHERE ANY(word IN all_keywords WHERE toLower(p.text) CONTAINS toLower(word))
    RETURN root.name AS term_name, COUNT(DISTINCT p) AS page_count
    """
    
    try:
        df_m = pd.DataFrame(session.run(query_mentions).data())
        df_p = pd.DataFrame(session.run(query_pages).data())
        session.close()

        df_comp = pd.merge(df_p, df_m, on='term_name')
        df_comp = df_comp.sort_values(by='page_count', ascending=False).head(10)
        df_comp_plot = df_comp.iloc[::-1] 

        lista_colores = [colores_tfg.get(x.lower(), '#333') for x in df_comp_plot['term_name']]

        # GRAFICA
        plt.figure(figsize=(16, 8), facecolor='none') 
        ax = plt.gca()
        ax.set_facecolor('none')
        
        y_pos = np.arange(len(df_comp_plot['term_name']))
        height = 0.35 

        bar_pages = plt.barh(y_pos + height/2, df_comp_plot['page_count'], height, 
                             color=lista_colores, hatch='///', edgecolor='black', label='Volumen (Páginas Únicas)')
        
        bar_mentions = plt.barh(y_pos - height/2, df_comp_plot['total_mentions'], height, 
                                color=lista_colores, alpha=0.6, edgecolor='black', label='Menciones (Total Relaciones)')

        plt.xscale('log')
        max_val = max(df_comp_plot['total_mentions'].max(), df_comp_plot['page_count'].max())
        plt.xlim(right=max_val * 8) 

        for i, bars in enumerate([bar_pages, bar_mentions]):
            for bar in bars:
                width = bar.get_width()
                if width > 0:
                    sufijo = " pags" if i == 0 else " menc."
                    plt.text(width * 1.15, bar.get_y() + bar.get_height()/2,
                             f'{int(width):,}{sufijo}', va='center', fontweight='bold', fontsize=10)

        plt.yticks(y_pos, df_comp_plot['term_name'], fontsize=12, fontweight='bold')
        for tick_label, color in zip(ax.get_yticklabels(), lista_colores):
            tick_label.set_color(color)

        plt.title('Comparativa: Volumen de Páginas vs Total de Menciones', fontsize=16, fontweight='bold', pad=30, color='#333')
        plt.xlabel('Cantidad (Escala Logarítmica)', fontsize=12)
        plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=2, shadow=True, frameon=True)
        plt.grid(axis='x', linestyle='--', alpha=0.3)
        sns.despine()
        plt.tight_layout()
        
        os.makedirs('Graficas', exist_ok=True)
        plt.savefig(os.path.join('Graficas', 'comparativa_final.svg'), dpi=300, bbox_inches='tight')
        plt.show()

        # TABLA 
        df_tabla_final = df_comp.copy().reset_index(drop=True)
        df_tabla_final.columns = ['Término', 'Páginas', 'Menciones']
        
        print("\n📊 TABLA COMPARATIVA COMPACTA:")
        
        display(HTML("""
            <style>
                .itables table { 
                    margin-left: 0 !important; 
                    margin-right: auto !important; 
                    width: 700px !important; 
                    max-width: 800px !important;
                    font-size: 0.85em !important;
                    border-collapse: collapse !important;
                    table-layout: fixed !important; 
                }
                .itables th, .itables td { 
                    padding: 10px 15px !important; 
                    white-space: nowrap !important;
                    overflow: hidden;
                    text-overflow: ellipsis;
                }
                .itables th { background-color: #f7fafc !important; border-bottom: 2px solid #edf2f7 !important; }
                .itables td { border-bottom: 1px solid #edf2f7 !important; }
            </style>
            """))

        show(df_tabla_final, 
             paging=False, 
             search=False, 
             info=False, 
             dom='t',
             autoWidth=False,
             columnDefs=[
                 {"width": "300px", "targets": 0}, # Término
                 {"width": "200px", "targets": 1}, # Páginas
                 {"width": "200px", "targets": 2}, # Menciones
                 {"className": "dt-left", "targets": "_all"}
             ])

    except Exception as e:
        print(f"❌ Error: {e}")

In [ ]:
# SILENCIAR
import itables.options as opt
opt.warn_on_undocumented_option = False
pd.options.mode.chained_assignment = None 

# CONEXION
if neo_driver:
    session = neo_driver.session()

    # CONSULTA
    query_mentions = """
    MATCH (t:Term)
    OPTIONAL MATCH (t)<-[:IS_SYNONYM_OF]-(s:Synonym)
    OPTIONAL MATCH (s)<-[:MENTIONS]-(p)
    RETURN t.name AS term_name, count(p) AS total_mentions
    """
    query_pages = """
    MATCH (root:Term)
    OPTIONAL MATCH (synonym:Synonym)-[:IS_SYNONYM_OF]->(root)
    WITH root, collect(toLower(synonym.name)) + toLower(root.name) AS all_keywords
    OPTIONAL MATCH (p:Page)
    WHERE ANY(word IN all_keywords WHERE toLower(p.text) CONTAINS toLower(word))
    RETURN root.name AS term_name, COUNT(DISTINCT p) AS page_count
    """
    
    try:
        df_m = pd.DataFrame(session.run(query_mentions).data())
        df_p = pd.DataFrame(session.run(query_pages).data())
        session.close()

        df_comp = pd.merge(df_p, df_m, on='term_name')
        df_comp = df_comp.sort_values(by='page_count', ascending=False).head(10)
        df_comp_plot = df_comp.iloc[::-1] 

        lista_colores = [colores_tfg.get(x.lower(), '#333') for x in df_comp_plot['term_name']]

        # =========================================================================
        # CONFIGURACIÓN DE GRÁFICA DE ALTA VISIBILIDAD (ESPECIAL POWERPOINT/PDF)
        # =========================================================================
        plt.figure(figsize=(18, 9), facecolor='none') 
        ax = plt.gca()
        ax.set_facecolor('none')
        
        y_pos = np.arange(len(df_comp_plot['term_name']))
        height = 0.35 

        bar_pages = plt.barh(y_pos + height/2, df_comp_plot['page_count'], height, 
                             color=lista_colores, hatch='///', edgecolor='black', 
                             label='Volumen (Páginas Únicas)')
        
        bar_mentions = plt.barh(y_pos - height/2, df_comp_plot['total_mentions'], height, 
                                color=lista_colores, alpha=0.6, edgecolor='black', 
                                label='Menciones (Total Relaciones)')

        plt.xscale('log')
        max_val = max(df_comp_plot['total_mentions'].max(), df_comp_plot['page_count'].max())
        plt.xlim(right=max_val * 12) # Aumentado el margen derecho para que quepan las etiquetas gigantes

        # ETIQUETAS DE LAS BARRAS (¡Más grandes!)
        for i, bars in enumerate([bar_pages, bar_mentions]):
            for bar in bars:
                width = bar.get_width()
                if width > 0:
                    sufijo = " págs." if i == 0 else " menc."
                    plt.text(width * 1.25, bar.get_y() + bar.get_height()/2,
                             f'{int(width):,}{sufijo}', va='center', fontweight='bold', 
                             fontsize=12, color='#222') # Subido a fontsize=12

        # EJES Y LETRAS LATERALES
        plt.yticks(y_pos, df_comp_plot['term_name'], fontsize=14, fontweight='bold') # Subido a fontsize=14
        plt.xticks(fontsize=13, fontweight='bold') # Numeración del eje X grande
        
        for tick_label, color in zip(ax.get_yticklabels(), lista_colores):
            tick_label.set_color(color)

        # TITULOS Y EJES
        plt.title('Comparativa: Volumen de Páginas vs Total de Menciones', 
                  fontsize=18, fontweight='bold', pad=35, color='#333') # Subido a fontsize=18
        plt.xlabel('Cantidad (Escala Logarítmica)', fontsize=14, fontweight='bold', labelpad=15) # Subido a fontsize=14
        
        # LEYENDA (¡Letras grandes y visibles desde atrás!)
        plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=2, 
                   shadow=True, frameon=True, fontsize=13, title_fontsize=14) # Subido a fontsize=13
        
        plt.grid(axis='x', linestyle='--', alpha=0.3)
        sns.despine()
        plt.tight_layout()
        
        # GUARDADO EN VECTORIAL .SVG
        os.makedirs('Graficas', exist_ok=True)
        plt.savefig(os.path.join('Graficas', 'comparativa_final.svg'), bbox_inches='tight')
        plt.show()

        # TABLA 
        df_tabla_final = df_comp.copy().reset_index(drop=True)
        df_tabla_final.columns = ['Término', 'Páginas', 'Menciones']
        
        print("\n📊 TABLA COMPARATIVA COMPACTA:")
        
        display(HTML("""
            <style>
                .itables table { 
                    margin-left: 0 !important; 
                    margin-right: auto !important; 
                    width: 700px !important; 
                    max-width: 800px !important;
                    font-size: 0.85em !important;
                    border-collapse: collapse !important;
                    table-layout: fixed !important; 
                }
                .itables th, .itables td { 
                    padding: 10px 15px !important; 
                    white-space: nowrap !important;
                    overflow: hidden;
                    text-overflow: ellipsis;
                }
                .itables th { background-color: #f7fafc !important; border-bottom: 2px solid #edf2f7 !important; }
                .itables td { border-bottom: 1px solid #edf2f7 !important; }
            </style>
            """))

        show(df_tabla_final, 
             paging=False, 
             search=False, 
             info=False, 
             dom='t',
             autoWidth=False,
             columnDefs=[
                 {"width": "300px", "targets": 0}, 
                 {"width": "200px", "targets": 1}, 
                 {"width": "200px", "targets": 2}, 
                 {"className": "dt-left", "targets": "_all"}
             ])

    except Exception as e:
        print(f"❌ Error: {e}")

### Media de menciones por página

Con esta gráfica se revisa la intensidad temática real, la media de menciones por página de cada término.

In [ ]:
# CONEXIÓN Y ANÁLISIS DE INTENSIDAD TEMÁTICA
if neo_driver:
    session = neo_driver.session()

    # CONSULTA: Cálculo de menciones reales por página
    query_media_real = """
    MATCH (root:Term)
    OPTIONAL MATCH (synonym:Synonym)-[:IS_SYNONYM_OF]->(root)
    WITH root, collect(toLower(synonym.name)) + toLower(root.name) AS all_keywords
    MATCH (p:Page)
    WHERE ANY(word IN all_keywords WHERE toLower(p.text) CONTAINS word)
    UNWIND all_keywords AS keyword
    WITH root.name AS term_name, p, keyword, toLower(p.text) AS doc_text
    WHERE doc_text CONTAINS keyword
    WITH term_name, p, 
         (size(doc_text) - size(replace(doc_text, keyword, ""))) / size(keyword) AS mentions_in_page
    RETURN term_name, 
           (sum(mentions_in_page) * 1.0) / count(DISTINCT p) AS avg_mentions,
           sum(mentions_in_page) AS total_mentions, 
           count(DISTINCT p) AS unique_pages
    ORDER BY avg_mentions DESC
    LIMIT 10
    """  
    try:
        records = session.run(query_media_real).data()
        df_avg = pd.DataFrame(records)
        session.close()

        if not df_avg.empty:
            df_plot = df_avg.copy()
            
            # Normalizamos nombres a minúsculas para buscar el color correctamente
            lista_colores = [colores_tfg.get(x.lower().strip(), '#333333') for x in df_plot['term_name']]

            # --- 1. GRÁFICA (CORREGIDA PARA EVITAR FUTUREWARNING) ---
            plt.figure(figsize=(14, 6), facecolor='none')
            ax = plt.gca()
            ax.set_facecolor('none')
            
            # Usamos hue= e identificamos la variable para cumplir con las nuevas normas de Seaborn
            plot = sns.barplot(
                data=df_plot, 
                x='avg_mentions', 
                y='term_name', 
                hue='term_name',       # Nueva norma de Seaborn
                palette=lista_colores, 
                legend=False,          # Evitamos leyenda duplicada
                edgecolor='black', 
                linewidth=1.5,
                alpha=0.85
            )
            
            # Añadir etiquetas de valor al final de las barras
            max_val = df_plot['avg_mentions'].max()
            for p in plot.patches:
                val = p.get_width()
                if val > 0:
                    plt.text(val + (max_val * 0.01), p.get_y() + p.get_height() / 2, 
                             f'{val:.2f}', va='center', fontweight='bold', fontsize=10)
            
            # Formatear etiquetas del eje Y (Categorías en mayúsculas y con color)
            ax.set_yticks(range(len(df_plot)))
            ax.set_yticklabels([str(n).upper() for n in df_plot['term_name']], fontweight='bold', fontsize=11)
            for tick_label, color in zip(ax.get_yticklabels(), lista_colores):
                tick_label.set_color(color)
                
            plt.title('Intensidad Temática: Media de Menciones por Página', fontsize=15, fontweight='bold', pad=25, color='#333')
            plt.xlabel('Promedio de apariciones por página (Densidad)', fontsize=10)
            plt.grid(axis='x', linestyle='--', alpha=0.3)
            plt.xlim(0, max_val * 1.15)
            sns.despine()
            plt.tight_layout()
            plt.show()

            # --- 2. TABLA (ALINEADA Y FORMATEADA) ---
            print("\n📊 DETALLE DE INTENSIDAD REAL:")
            
            # Preparamos el DF para la tabla
            df_tabla = df_avg.copy()
            df_tabla['term_name'] = df_tabla['term_name'].str.upper()
            df_tabla['avg_mentions'] = df_tabla['avg_mentions'].round(2)
            df_tabla.columns = ['Categoría', 'Media (Densidad)', 'Total Menc.', 'Págs Únicas']

            # Inyectamos CSS para forzar la alineación a la izquierda en Jupyter
            display(HTML("""
            <style>
                .jp-RenderedHTMLCommon table, .rendered_html table {
                    margin-left: 0 !important;
                    margin-right: auto !important;
                }
                .itables table { 
                    margin-left: 0 !important; 
                    width: 750px !important; 
                    font-size: 0.85em !important;
                }
                .itables th, .itables td { text-align: left !important; padding: 8px 12px !important; }
            </style>
            """))

            show(df_tabla, paging=False, search=False, info=False, dom='t', autoWidth=False)

            # --- 3. GUARDADO Y DESCARGA ---
            ruta_dir = 'Graficas'
            os.makedirs(ruta_dir, exist_ok=True)
            nombre_archivo = "intensidad_tematica_real.csv"
            path_completo = os.path.join(ruta_dir, nombre_archivo)
            df_tabla.to_csv(path_completo, index=False)

            csv_data = df_tabla.to_csv(index=False)
            b64_csv = base64.b64encode(csv_data.encode()).decode()
            payload = f"data:text/csv;base64,{b64_csv}"
            
            display(HTML(f"""
            <div style="text-align: left; margin-top: 15px; margin-bottom: 20px;">
                <span style="font-size: 10px; color: #718096; display: block; margin-bottom: 8px;">📁 Archivo: {path_completo}</span>
                <a href="{payload}" download="{nombre_archivo}" style="
                    display: inline-block; background-color: #4299e1; color: white;
                    padding: 6px 14px; text-decoration: none; border-radius: 6px;
                    font-weight: 600; font-family: sans-serif; font-size: 12px;
                ">📥 Descargar CSV de Intensidad</a>
            </div>
            """))

    except Exception as e:
        print(f"❌ Error durante el análisis: {e}")

### Análisis por volumen de páginas únicas

Con esta gráfica se ven la frecuencía de aparición por páginas únicas de los sinónimos de un término dado (en este caso DRUGS).  
Ese termino raíz puede ser cambiado por cualquiera de los otros terminos raíz para poder realizar un estudio del resto de terminos, se realizaría con: target = "termino_raiz"

In [ ]:
# SILENCIAR
opt.warn_on_undocumented_option = False

# CONSULTA
def get_page_count_data(term_name, neo_driver):
    query = """
    MATCH (root:Term {name: $name})
    OPTIONAL MATCH (synonym:Synonym)-[:IS_SYNONYM_OF]->(root)
    WITH root, collect(synonym) AS synonyms
    WITH [root] + synonyms AS all_terms
    UNWIND all_terms AS t
    WITH t WHERE t.name IS NOT NULL
    
    OPTIONAL MATCH (t)<-[:MENTIONS]-(p:Page)
    
    RETURN 
      t.name AS term_name,
      CASE WHEN "Term" IN labels(t) THEN "Raíz" ELSE "Sinónimo" END AS type,
      COUNT(DISTINCT p) AS page_count
    ORDER BY page_count DESC
    """
    try:
        with neo_driver.session() as session:
            result = session.run(query, name=term_name)
            data = [dict(record) for record in result]
            
            df_raw = pd.DataFrame(data)
            if not df_raw.empty:
                df_raw = df_raw.sort_values('page_count', ascending=False).drop_duplicates('term_name')
            return df_raw
    except Exception as e:
        print(f"❌ Error en query: {e}")
        return pd.DataFrame()

# AQUI SE CAMBIA EL TERMINO PARA PODER REALIZAR UN ANALISIS DE TODOS LOS TERMINOS.
target = "drugs" 
df = get_page_count_data(target, neo_driver)

if not df.empty and (df['page_count'].sum() > 0):
    df_plot = df.sort_values(by='page_count', ascending=False) 
    total_paginas = df_plot['page_count'].sum()
    color_actual = colores_tfg.get(target.lower(), '#333') 
    
    # GRAFICA
    plt.figure(figsize=(14, 5), facecolor='none') 
    ax = plt.gca()
    ax.set_facecolor('none')
    
    plot = sns.barplot(
        data=df_plot, 
        x='page_count', 
        y='term_name', 
        color=color_actual, 
        edgecolor='black', 
        linewidth=1.5,
        alpha=0.85
    )

    max_val = df_plot['page_count'].max()
    for p in plot.patches:
        width = p.get_width()
        if width > 0:
            pct = (width / total_paginas * 100)
            plt.text(width + (max_val * 0.01), p.get_y() + p.get_height() / 2, 
                     f'{int(width)} ({pct:.1f}%)', va='center', fontweight='bold', fontsize=10)

    for label in ax.get_yticklabels():
        label.set_fontweight('bold')
        label.set_fontsize(11)
        label.set_color(color_actual) 

    plt.title(f'Presencia en Dark Web: {target.upper()}', fontsize=15, fontweight='bold', pad=25, color='#333')
    plt.xlabel('Número de Páginas Únicas', fontsize=10)
    plt.xlim(0, max_val * 1.2)
    plt.grid(axis='x', linestyle='--', alpha=0.3)
    sns.despine()
    plt.tight_layout()
    plt.show()

    # TABLA
    # --- TABLA CORREGIDA ---
    df_tabla = df_plot.copy()
    df_tabla['%'] = (df_tabla['page_count'] / total_paginas * 100).round(2)
    df_tabla.columns = ['Término', 'Tipo', 'Páginas', 'Porcentaje (%)']
    df_tabla = df_tabla.reset_index(drop=True)
    
    print(f"\n📊 REPORTE DE PRESENCIA: {target.upper()}")
    
    # CSS Forzado a la izquierda (idéntico a tus otras gráficas)
    display(HTML("""
            <style>
                .jp-RenderedHTMLCommon table, .rendered_html table {
                    margin-left: 0 !important;
                    margin-right: auto !important;
                }
                .itables {
                    text-align: left !important;
                    margin-left: 0 !important;
                }
                .itables table { 
                    margin-left: 0 !important; 
                    margin-right: auto !important; 
                    width: 700px !important; 
                    max-width: 800px !important;
                    font-size: 0.85em !important;
                    border-collapse: collapse !important;
                    table-layout: fixed !important; 
                }
                .itables th, .itables td { 
                    padding: 10px 15px !important; 
                    text-align: left !important;
                    white-space: nowrap !important; 
                    overflow: hidden;
                    text-overflow: ellipsis;
                }
                .itables th { background-color: #f7fafc !important; border-bottom: 2px solid #edf2f7 !important; }
                .itables td { border-bottom: 1px solid #edf2f7 !important; }
            </style>
            """))

    # CAMBIO CLAVE: Cambiamos df_avg por df_tabla
    show(df_tabla, 
         paging=False, 
         search=False, 
         info=False, 
         dom='t',
         autoWidth=False, 
         columnDefs=[
             {"width": "200px", "targets": 0}, 
             {"width": "100px", "targets": 1}, 
             {"width": "150px", "targets": 2}, 
             {"width": "150px", "targets": 3}, 
             {"className": "dt-left", "targets": "_all"}
         ])

    # GUARDADO
    ruta_dir = 'Graficas'
    os.makedirs(ruta_dir, exist_ok=True)
    nombre_archivo = f"presencia_paginas_{target}.csv"
    path_completo = os.path.join(ruta_dir, nombre_archivo)
    df_tabla.to_csv(path_completo, index=False)

    csv_data = df_tabla.to_csv(index=False)
    b64_csv = base64.b64encode(csv_data.encode()).decode()
    payload = f"data:text/csv;base64,{b64_csv}"
    
    display(HTML(f"""
    <div style="text-align: left; margin-top: 15px;">
        <span style="font-size: 10px; color: #718096; display: block; margin-bottom: 8px;">📁 Guardado en: {path_completo}</span>
        <a href="{payload}" download="{nombre_archivo}" style="
            display: inline-block; background-color: {color_actual}; color: white;
            padding: 6px 14px; text-decoration: none; border-radius: 6px;
            font-weight: 600; font-family: sans-serif; font-size: 12px;
        ">📥 Descargar Reporte {target.upper()}</a>
    </div>
    """))
else:
    print(f"⚠️ No se han encontrado páginas para el término '{target}'.")

Con esta gráfica se ven la frecuencía de aparición por páginas de los sinónimos de un término dado (en este caso DRUGS)  
Al anterior código se le añaden unos filtros:
- Únicamente coger la x cuando está de manera aislada, con una coma o un punto.
- Un filtro de co-ocurrencia, es decir, solo cuente la x si en la misma página aparecen otras palabras clave del mundo de las drogas.

In [ ]:
# SILENCIAR
opt.warn_on_undocumented_option = False

def get_page_count_cleaned(term_name, neo_driver):
    # TERMINOS PARA EL FILTRADO DE CO-OCURRENCIA
    context_keywords = ["pills", "vendor", "mdma", "shipping", "buy", "order", "price", "quality", "ecstasy", "tabs", "market"]

    # CONSULTA
    query = """
    MATCH (root:Term {name: $name})
    OPTIONAL MATCH (synonym:Synonym)-[:IS_SYNONYM_OF]->(root)
    WITH root, collect(synonym) AS synonyms
    WITH [root] + synonyms AS all_terms
    UNWIND all_terms AS t
    WITH t WHERE t.name IS NOT NULL
    MATCH (t)<-[:MENTIONS]-(p:Page)
    RETURN 
      t.name AS term_name, 
      CASE WHEN "Term" IN labels(t) THEN "Raíz" ELSE "Sinónimo" END AS type, 
      p.text AS page_text
    """
    
    try:
        with neo_driver.session() as session:
            result = session.run(query, name=term_name)
            raw_data = [dict(record) for record in result]
            
        if not raw_data:
            return pd.DataFrame()

        processed_records = []
        for record in raw_data:
            name = record['term_name']
            text = record['page_text'] if record['page_text'] else ""
            text_lower = text.lower()
            
            if len(name) <= 2:
                is_isolated = re.search(rf"\b{re.escape(name)}\b", text, re.IGNORECASE)
                has_context = any(word in text_lower for word in context_keywords)
                if is_isolated and has_context:
                    processed_records.append(record)
            else:
                processed_records.append(record)
        
        df_raw = pd.DataFrame(processed_records)
        if df_raw.empty: return pd.DataFrame()
        
        df_final = df_raw.groupby(['term_name', 'type']).size().reset_index(name='page_count')
        df_final = df_final.sort_values('page_count', ascending=False).drop_duplicates('term_name')
        return df_final
        
    except Exception as e:
        print(f"❌ Error en la query: {e}")
        return pd.DataFrame()

# TERMINO A BUSCAR
target = "drugs" 
df = get_page_count_cleaned(target, neo_driver)

# GRAFICA
if not df.empty:
    df_plot = df.sort_values(by='page_count', ascending=False)
    total_paginas = df_plot['page_count'].sum()
    color_actual = colores_tfg.get(target.lower(), '#333')

    plt.figure(figsize=(14, 5), facecolor='none')
    ax = plt.gca()
    ax.set_facecolor('none')
    
    plot = sns.barplot(
        data=df_plot, 
        x='page_count', 
        y='term_name', 
        color=color_actual, 
        edgecolor='black', 
        linewidth=1.5,
        alpha=0.85
    )
    
    max_val = df_plot['page_count'].max()
    for p in plot.patches:
        width = p.get_width()
        if width > 0:
            pct = (width / total_paginas * 100) if total_paginas > 0 else 0
            plt.text(width + (max_val * 0.01), p.get_y() + p.get_height() / 2, 
                     f'{int(width)} ({pct:.1f}%)', 
                     va='center', fontweight='bold', fontsize=10)

    for label in ax.get_yticklabels():
        label.set_fontweight('bold')
        label.set_fontsize(11)
        label.set_color(color_actual)

    plt.title(f'Presencia Validada: {target.upper()} (Filtrado Semántico)', 
              fontsize=15, fontweight='bold', pad=25, color='#333')
    
    plt.xlabel('Páginas Únicas Detectadas', fontsize=10)
    plt.xlim(0, max_val * 1.2)
    plt.grid(axis='x', linestyle='--', alpha=0.3)
    sns.despine()
    plt.tight_layout()
    plt.show()

    # TABLA
    # --- TABLA CORREGIDA ---
    df_tabla = df_plot.copy()
    df_tabla['%'] = (df_tabla['page_count'] / total_paginas * 100).round(2)
    df_tabla.columns = ['Término', 'Tipo', 'Páginas', 'Porcentaje (%)']
    df_tabla = df_tabla.reset_index(drop=True)

    print(f"\n📊 REPORTE DE PÁGINAS VALIDADAS: {target.upper()}")
    
    # CSS Unificado y forzado a la izquierda
    display(HTML("""
            <style>
                .jp-RenderedHTMLCommon table, .rendered_html table {
                    margin-left: 0 !important;
                    margin-right: auto !important;
                }
                .itables {
                    text-align: left !important;
                    margin-left: 0 !important;
                }
                .itables table { 
                    margin-left: 0 !important; 
                    margin-right: auto !important; 
                    width: 700px !important; 
                    max-width: 800px !important;
                    font-size: 0.85em !important;
                    border-collapse: collapse !important;
                    table-layout: fixed !important; 
                }
                .itables th, .itables td { 
                    padding: 10px 15px !important; 
                    text-align: left !important;
                    white-space: nowrap !important; 
                    overflow: hidden;
                    text-overflow: ellipsis;
                }
                .itables th { background-color: #f7fafc !important; border-bottom: 2px solid #edf2f7 !important; }
                .itables td { border-bottom: 1px solid #edf2f7 !important; }
            </style>
            """))

    # CAMBIO: Usamos df_tabla que es el objeto real
    show(df_tabla, 
         paging=False, 
         search=False, 
         info=False, 
         dom='t',
         autoWidth=False, 
         columnDefs=[
             {"width": "200px", "targets": 0}, 
             {"width": "100px", "targets": 1}, 
             {"width": "150px", "targets": 2}, 
             {"width": "150px", "targets": 3}, 
             {"className": "dt-left", "targets": "_all"}
         ])

    # GUARDADO
    ruta_dir = 'Graficas'
    os.makedirs(ruta_dir, exist_ok=True)
    nombre_archivo = f"presencia_paginas_filtrada_{target}.csv"
    path_completo = os.path.join(ruta_dir, nombre_archivo)
    df_tabla.to_csv(path_completo, index=False)

    csv_data = df_tabla.to_csv(index=False)
    b64_csv = base64.b64encode(csv_data.encode()).decode()
    payload = f"data:text/csv;base64,{b64_csv}"
    
    display(HTML(f"""
    <div style="text-align: left; margin-top: 15px;">
        <span style="font-size: 10px; color: #718096; display: block; margin-bottom: 8px;">📁 Guardado en: {path_completo}</span>
        <a href="{payload}" download="{nombre_archivo}" style="
            display: inline-block; background-color: {color_actual}; color: white;
            padding: 6px 14px; text-decoration: none; border-radius: 6px;
            font-weight: 600; font-family: sans-serif; font-size: 12px;
        ">📥 Descargar Reporte Validado</a>
    </div>
    """))
else:
    print(f"⚠️ No se encontraron datos validados para '{target}'.")

### Análisis por frecuencia de menciones

Con esta gráfica se ve la frecuencía de aparición por menciones de los sinónimos de un término dado (en este caso DRUGS)  
Ese termino raíz puede ser cambiado por cualquiera de los otros terminos raíz para poder realizar un estudio del resto de terminos, se realizaría con: target = "termino_raiz"

In [ ]:
# SILENCIAR
opt.warn_on_undocumented_option = False

def get_data_by_text_search(term_name, neo_driver):
    # CONSULTA
    query = """
    MATCH (root:Term {name: $name})
    OPTIONAL MATCH (synonym:Synonym)-[:IS_SYNONYM_OF]->(root)
    WITH root, collect(synonym) AS synonyms
    WITH [root] + synonyms AS all_terms
    UNWIND all_terms AS t
    
    OPTIONAL MATCH (p:Page)
    WHERE toLower(p.text) CONTAINS toLower(t.name) 
       OR toLower(p.title) CONTAINS toLower(t.name)
    
    RETURN 
      t.name AS term_name,
      CASE WHEN "Term" IN labels(t) THEN "Raíz" ELSE "Sinónimo" END AS type,
      COUNT(DISTINCT p) AS mentions_count
    ORDER BY mentions_count DESC
    """
    try:
        with neo_driver.session() as session:
            result = session.run(query, name=term_name)
            data = [dict(record) for record in result]
            df_raw = pd.DataFrame(data)
            if not df_raw.empty:
                df_raw = df_raw.sort_values('mentions_count', ascending=False).drop_duplicates('term_name')
            return df_raw
    except Exception as e:
        print(f"❌ Error en la consulta: {e}")
        return pd.DataFrame()

# TERMINO QUE PUEDE SER CAMBIADO
target_term = "drugs" 
df = get_data_by_text_search(target_term, neo_driver)

if not df.empty and df['mentions_count'].sum() > 0:
    df_plot = df.sort_values(by='mentions_count', ascending=False)
    total_paginas = df_plot['mentions_count'].sum()
    color_actual = colores_tfg.get(target_term.lower(), '#333')
    
    # GRAFICA
    plt.figure(figsize=(14, 5), facecolor='none') 
    ax = plt.gca()
    ax.set_facecolor('none')
    
    plot = sns.barplot(
        data=df_plot, 
        x='mentions_count', 
        y='term_name', 
        color=color_actual, 
        edgecolor='black', 
        linewidth=1.5,
        alpha=0.85
    )
    
    max_val = df_plot['mentions_count'].max()
    for p in plot.patches:
        width = p.get_width()
        if width > 0:
            pct = (width / total_paginas * 100) if total_paginas > 0 else 0
            plt.text(width + (max_val * 0.01), p.get_y() + p.get_height() / 2, 
                     f'{int(width)} ({pct:.1f}%)', 
                     va='center', fontweight='bold', fontsize=10)

    for label in ax.get_yticklabels():
        label.set_fontweight('bold')
        label.set_fontsize(11)
        label.set_color(color_actual)

    plt.title(f'Frecuencia de Aparición: {target_term.upper()}', fontsize=15, fontweight='bold', pad=25, color='#333')
    plt.xlabel('Número de menciones detectadas', fontsize=10)
    plt.xlim(0, max_val * 1.2)
    plt.grid(axis='x', linestyle='--', alpha=0.3)
    sns.despine()
    plt.tight_layout()
    plt.show()

    # TABLA
    # TABLA CORREGIDA
    df_resumen = df_plot.copy()
    df_resumen['%'] = (df_resumen['mentions_count'] / total_paginas * 100).round(2)
    df_resumen.columns = ['Término', 'Tipo', 'Menciones', 'Porcentaje (%)']
    df_resumen = df_resumen.reset_index(drop=True)

    print(f"\n📊 RESULTADOS DE BÚSQUEDA: {target_term.upper()}")
    
    # CSS para forzar la alineación a la izquierda
    display(HTML("""
            <style>
                .itables table { 
                    margin-left: 0 !important; 
                    margin-right: auto !important; 
                    width: 700px !important; 
                    max-width: 800px !important;
                    font-size: 0.85em !important;
                    border-collapse: collapse !important;
                    table-layout: fixed !important; 
                }
                .itables th, .itables td { 
                    padding: 10px 15px !important; 
                    text-align: left !important;
                    white-space: nowrap !important; 
                    overflow: hidden;
                    text-overflow: ellipsis;
                }
                .itables th { background-color: #f7fafc !important; border-bottom: 2px solid #edf2f7 !important; }
                .itables td { border-bottom: 1px solid #edf2f7 !important; }
            </style>
            """))

    # CAMBIO CLAVE: Usamos df_resumen en lugar de df_avg
    show(df_resumen, 
         paging=False, 
         search=False, 
         info=False, 
         dom='t',
         autoWidth=False, 
         columnDefs=[
             {"width": "200px", "targets": 0}, # Término
             {"width": "100px", "targets": 1}, # Tipo
             {"width": "150px", "targets": 2}, # Menciones
             {"width": "150px", "targets": 3}, # Porcentaje
             {"className": "dt-left", "targets": "_all"}
         ])
    
    # GUARDADO
    ruta_dir = 'Graficas'
    os.makedirs(ruta_dir, exist_ok=True)
    nombre_archivo = f"frecuencia_aparicion_texto_{target_term}.csv"
    path_completo = os.path.join(ruta_dir, nombre_archivo)
    df_resumen.to_csv(path_completo, index=False)
    
    csv_data = df_resumen.to_csv(index=False)
    b64_csv = base64.b64encode(csv_data.encode()).decode()
    payload = f"data:text/csv;base64,{b64_csv}"
    
    display(HTML(f"""
    <div style="text-align: left; margin-top: 15px;">
        <span style="font-size: 10px; color: #718096; display: block; margin-bottom: 8px;">📁 Guardado en: {path_completo}</span>
        <a href="{payload}" download="{nombre_archivo}" style="
            display: inline-block; background-color: {color_actual}; color: white;
            padding: 6px 14px; text-decoration: none; border-radius: 6px;
            font-weight: 600; font-family: sans-serif; font-size: 12px;
        ">📥 Descargar Reporte {target_term.upper()}</a>
    </div>
    """))
else:
    print(f"⚠️ No se han encontrado páginas con el texto '{target_term}'.")

Con esta gráfica se ven la frecuencía de aparición por menciones de los sinónimos de un término dado (en este caso DRUGS).  
Al anterior código se le añaden unos filtros:
- Únicamente coger la x cuando está de manera aislada, con una coma o un punto.
- Un filtro de co-ocurrencia, es decir, solo cuente la x si en la misma página aparecen otras palabras clave del mundo de las drogas.

In [ ]:
# SILENCIAR
opt.warn_on_undocumented_option = False

def get_data_by_text_search_cleaned(term_name, neo_driver):
    # PALABRAS PARA EL FILTRADO DE CO-OCURRENCIA
    context_keywords = ["pills", "vendor", "mdma", "shipping", "buy", "order", "price", "quality", "ecstasy", "tabs"]

    # CONSULTA
    query = """
    MATCH (root:Term {name: $name})
    OPTIONAL MATCH (synonym:Synonym)-[:IS_SYNONYM_OF]->(root)
    WITH root, collect(synonym) AS synonyms
    WITH [root] + synonyms AS all_terms
    UNWIND all_terms AS t
    
    OPTIONAL MATCH (p:Page)
    WHERE (
        CASE 
            WHEN size(t.name) <= 2 THEN 
                (p.text =~ ("(?i).*\\\\b" + t.name + "\\\\b.*") OR p.title =~ ("(?i).*\\\\b" + t.name + "\\\\b.*"))
                AND
                ANY(word IN $context WHERE toLower(p.text) CONTAINS word OR toLower(p.title) CONTAINS word)
            ELSE 
                toLower(p.text) CONTAINS toLower(t.name) OR 
                toLower(p.title) CONTAINS toLower(t.name)
        END
    )
    
    RETURN 
      t.name AS term_name,
      CASE WHEN "Term" IN labels(t) THEN "Raíz" ELSE "Sinónimo" END AS type,
      COUNT(DISTINCT p) AS mentions_count
    ORDER BY mentions_count DESC
    """
    try:
        with neo_driver.session() as session:
            result = session.run(query, name=term_name, context=context_keywords)
            data = [dict(record) for record in result]
            df_raw = pd.DataFrame(data)
            if not df_raw.empty:
                df_raw = df_raw.sort_values('mentions_count', ascending=False).drop_duplicates('term_name')
            return df_raw
    except Exception as e:
        print(f"❌ Error en la consulta: {e}")
        return pd.DataFrame()

# TERMINO ELEGIDO
target_term = "drugs" 
df = get_data_by_text_search_cleaned(target_term, neo_driver)

if not df.empty and df['mentions_count'].sum() > 0:
    df_plot = df.sort_values(by='mentions_count', ascending=False)
    total_paginas = df_plot['mentions_count'].sum()
    color_actual = colores_tfg.get(target_term.lower(), '#333')
    
    # GRAFICA
    plt.figure(figsize=(14, 5), facecolor='none') 
    ax = plt.gca()
    ax.set_facecolor('none')
    
    plot = sns.barplot(
        data=df_plot, 
        x='mentions_count', 
        y='term_name', 
        color=color_actual, 
        edgecolor='black', 
        linewidth=1.5,
        alpha=0.85
    )
    
    max_val = df_plot['mentions_count'].max()
    for p in plot.patches:
        width = p.get_width()
        if width > 0:
            pct = (width / total_paginas * 100) if total_paginas > 0 else 0
            plt.text(width + (max_val * 0.01), p.get_y() + p.get_height() / 2, 
                     f'{int(width)} ({pct:.1f}%)', 
                     va='center', fontweight='bold', fontsize=10)

    for label in ax.get_yticklabels():
        label.set_fontweight('bold')
        label.set_fontsize(11)
        label.set_color(color_actual)

    plt.title(f'Contexto Validado (Semántico): {target_term.upper()}', fontsize=15, fontweight='bold', pad=25, color='#333')
    plt.xlabel('Menciones con contexto validado', fontsize=10)
    plt.xlim(0, max_val * 1.2)
    plt.grid(axis='x', linestyle='--', alpha=0.3)
    sns.despine()
    plt.tight_layout()
    plt.show()

    # TABLA
    # TABLA CORREGIDA
    df_resumen = df_plot.copy()
    df_resumen['%'] = (df_resumen['mentions_count'] / total_paginas * 100).round(2)
    df_resumen.columns = ['Término', 'Tipo', 'Menciones', 'Porcentaje (%)']
    df_resumen = df_resumen.reset_index(drop=True)

    print(f"\n📊 RESULTADOS DE BÚSQUEDA: {target_term.upper()}")
    
    # CSS para forzar la alineación a la izquierda
    display(HTML("""
            <style>
                .itables table { 
                    margin-left: 0 !important; 
                    margin-right: auto !important; 
                    width: 700px !important; 
                    max-width: 800px !important;
                    font-size: 0.85em !important;
                    border-collapse: collapse !important;
                    table-layout: fixed !important; 
                }
                .itables th, .itables td { 
                    padding: 10px 15px !important; 
                    text-align: left !important;
                    white-space: nowrap !important; 
                    overflow: hidden;
                    text-overflow: ellipsis;
                }
                .itables th { background-color: #f7fafc !important; border-bottom: 2px solid #edf2f7 !important; }
                .itables td { border-bottom: 1px solid #edf2f7 !important; }
            </style>
            """))

    # CAMBIO CLAVE: Usamos df_resumen en lugar de df_avg
    show(df_resumen, 
         paging=False, 
         search=False, 
         info=False, 
         dom='t',
         autoWidth=False, 
         columnDefs=[
             {"width": "200px", "targets": 0}, # Término
             {"width": "100px", "targets": 1}, # Tipo
             {"width": "150px", "targets": 2}, # Menciones
             {"width": "150px", "targets": 3}, # Porcentaje
             {"className": "dt-left", "targets": "_all"}
         ])

    # GUARDADO
    ruta_dir = 'Graficas'
    os.makedirs(ruta_dir, exist_ok=True)
    nombre_archivo = f"frecuencia_aparicion_filtrada_{target_term}.csv"
    path_completo = os.path.join(ruta_dir, nombre_archivo)
    df_resumen.to_csv(path_completo, index=False)
    
    csv_data = df_resumen.to_csv(index=False)
    b64_csv = base64.b64encode(csv_data.encode()).decode()
    payload = f"data:text/csv;base64,{b64_csv}"
    
    display(HTML(f"""
    <div style="text-align: left; margin-top: 15px;">
        <span style="font-size: 10px; color: #718096; display: block; margin-bottom: 8px;">📁 Guardado en: {path_completo}</span>
        <a href="{payload}" download="{nombre_archivo}" style="
            display: inline-block; background-color: {color_actual}; color: white;
            padding: 6px 14px; text-decoration: none; border-radius: 6px;
            font-weight: 600; font-family: sans-serif; font-size: 12px;
        ">📥 Descargar Reporte Validado</a>
    </div>
    """))
else:
    print(f"⚠️ No se han encontrado datos para '{target_term}'.")

### Top 20 términos

Aquí se realiza un top 20 de términos y sus sinónimos por la cantidad de menciones.

In [ ]:
def run_analysis_colored_fix(limit=200):
    print("-> Accediendo a Neo4j para mapear Raíces...")
    term_to_root = {}
    root_terms = set()
    
    try:
        with GraphDatabase.driver(NEO_URI, auth=(NEO_USER, NEO_PASS)) as driver:
            with driver.session() as session:
                query = """
                MATCH (t:Term)
                OPTIONAL MATCH (s:Synonym)-[:IS_SYNONYM_OF]->(t)
                RETURN t.name as root, s.name as synonym
                """
                results = session.run(query)
                for record in results:
                    root = record["root"].lower()
                    syn = record["synonym"]
                    root_terms.add(root)
                    term_to_root[root] = root 
                    if syn: 
                        term_to_root[syn.lower()] = root 
    except Exception as e:
        print(f"❌ Error en Neo4j: {e}")
        return

    # MONGODB
    client = MongoClient(MONGO_URI)
    db = client[MONGO_DB_NAME]
    fs = GridFS(db)
    
    files = list(db['fs.files'].find().limit(limit))
    term_counts = {t: 0 for t in term_to_root.keys()}
    # REGEX PARA LIMITES DE PALABRA, ES DECIR, EVITAR QUE TENGA DRUG EN VEZ DE DRUGS...
    patterns = {t: re.compile(r'\b' + re.escape(t) + r'\b', re.IGNORECASE) for t in term_to_root.keys()}

    print(f"-> Procesando texto de {len(files)} documentos HTML...")
    for f in tqdm(files):
        try:
            txt = fs.get(f['_id']).read().decode('utf-8', errors='ignore')
            clean = BeautifulSoup(txt, "html.parser").get_text().lower()
            for t, p in patterns.items():
                matches = len(p.findall(clean))
                if matches > 0: 
                    term_counts[t] += matches
        except: 
            continue

    # PROCESAR DATOS
    df = pd.DataFrame(list(term_counts.items()), columns=['Termino', 'Frecuencia'])
    df = df[df['Frecuencia'] > 0].sort_values('Frecuencia', ascending=False).head(20)
    
    if df.empty: 
        return print("❌ Sin datos suficientes para graficar.")

    # COLORES
    lista_colores = []
    roots_en_grafica = set() 
    for term in df['Termino']:
        padre = term_to_root.get(term.lower(), term.lower())
        color = colores_tfg.get(padre, '#999999')
        lista_colores.append(color)
        roots_en_grafica.add(padre)

    # GRAFICA
    plt.figure(figsize=(14, 9), facecolor='white') # Fondo blanco para claridad
    ax = plt.gca()
    plot = sns.barplot(
        data=df, x='Frecuencia', y='Termino', 
        palette=lista_colores, hue='Termino',
        legend=False, edgecolor='black', linewidth=1.5, alpha=0.85
    )

    patches = [mpatches.Patch(color=colores_tfg.get(r, '#999'), label=r.upper()) for r in sorted(list(roots_en_grafica))]
    
    leg = plt.legend(
        handles=patches, 
        title="Categoría Raíz (Semántica)", 
        bbox_to_anchor=(0.5, 1.15),  # Bajada un poco para dar aire al título
        loc='upper center', 
        ncol=min(len(patches), 4), 
        frameon=True, fontsize=9, shadow=True
    )
    plt.setp(leg.get_title(), fontweight='bold')
    plt.title('Análisis de Frecuencia: Términos Detectados por Ontología', 
              fontsize=16, fontweight='bold', pad=65, color='#333')
    
    # ETIQUETAS
    max_val = df['Frecuencia'].max()
    for p in plot.patches:
        width = p.get_width()
        if width > 0:
            plt.text(width + (max_val * 0.01), p.get_y() + p.get_height()/2, 
                     f'{int(width)}', va='center', fontsize=10, fontweight='bold')

    for i, label in enumerate(ax.get_yticklabels()):
        label.set_fontweight('bold')
        label.set_color(lista_colores[i])

    plt.xlabel('Número total de menciones', fontsize=11)
    plt.ylabel('') 
    plt.grid(axis='x', linestyle='--', alpha=0.3)
    sns.despine()
    
    plt.subplots_adjust(top=0.78)  

    # GUARDADO
    ruta_dir = 'Graficas'
    os.makedirs(ruta_dir, exist_ok=True)
    plt.savefig(os.path.join(ruta_dir, 'frecuencia_ontologia_final.svg'), dpi=300, bbox_inches='tight')
    plt.show()

    # TABLA
    print("\n📊 DETALLE DE FRECUENCIAS DETECTADAS:")
    df_tabla = df.copy()
    total_m = df_tabla['Frecuencia'].sum()
    df_tabla['%'] = (df_tabla['Frecuencia'] / total_m * 100).round(2)
    df_tabla.columns = ['Nombre', 'Menciones', 'Porcentaje (%)']
    
    display(HTML("""
        <style>
            .itables table { 
                margin-left: 0 !important; 
                margin-right: auto !important; 
                width: 700px !important;
                font-size: 0.85em !important;
                border-collapse: collapse !important;
            }
            .itables th { background-color: #f7fafc !important; border-bottom: 2px solid #edf2f7 !important; text-align: left !important;}
            .itables td { border-bottom: 1px solid #edf2f7 !important; text-align: left !important;}
        </style>
    """))
    show(df_tabla, 
         paging=False, 
         search=False, 
         info=False, 
         dom='t',
         columnDefs=[
             {"width": "250px", "targets": 0}, 
             {"width": "150px", "targets": 1}, 
             {"width": "150px", "targets": 2}, 
             {"className": "dt-left", "targets": "_all"}
         ])

    # GUARDADO
    csv_data = df_tabla.to_csv(index=False)
    b64_csv = base64.b64encode(csv_data.encode()).decode()
    payload = f"data:text/csv;base64,{b64_csv}"
    
    display(HTML(f"""
    <div style="text-align: left; margin-top: 15px;">
        <span style="font-size: 10px; color: #718096; display: block; margin-bottom: 8px;">📁 Guardado en: {ruta_dir}/frecuencia_ontologia_final.svg</span>
        <a href="{payload}" download="frecuencia_ontologia.csv" style="
            display: inline-block; background-color: #4A5568; color: white;
            padding: 6px 14px; text-decoration: none; border-radius: 6px;
            font-weight: 600; font-family: sans-serif; font-size: 12px;
        ">📥 Descargar Frecuencias CSV</a>
    </div>
    """))

# EJECUTAR CODIGO
run_analysis_colored_fix(limit=300)

In [ ]:
def run_analysis_colored_fix(limit=200):
    print("-> Accediendo a Neo4j para mapear Raíces...")
    term_to_root = {}
    root_terms = set()
    
    try:
        with GraphDatabase.driver(NEO_URI, auth=(NEO_USER, NEO_PASS)) as driver:
            with driver.session() as session:
                query = """
                MATCH (t:Term)
                OPTIONAL MATCH (s:Synonym)-[:IS_SYNONYM_OF]->(t)
                RETURN t.name as root, s.name as synonym
                """
                results = session.run(query)
                for record in results:
                    root = record["root"].lower()
                    syn = record["synonym"]
                    root_terms.add(root)
                    term_to_root[root] = root 
                    if syn: 
                        term_to_root[syn.lower()] = root 
    except Exception as e:
        print(f"❌ Error en Neo4j: {e}")
        return

    # MONGODB
    client = MongoClient(MONGO_URI)
    db = client[MONGO_DB_NAME]
    fs = GridFS(db)
    
    files = list(db['fs.files'].find().limit(limit))
    term_counts = {t: 0 for t in term_to_root.keys()}
    # REGEX PARA LIMITES DE PALABRA, ES DECIR, EVITAR QUE TENGA DRUG EN VEZ DE DRUGS...
    patterns = {t: re.compile(r'\b' + re.escape(t) + r'\b', re.IGNORECASE) for t in term_to_root.keys()}

    print(f"-> Procesando texto de {len(files)} documentos HTML...")
    for f in tqdm(files):
        try:
            txt = fs.get(f['_id']).read().decode('utf-8', errors='ignore')
            clean = BeautifulSoup(txt, "html.parser").get_text().lower()
            for t, p in patterns.items():
                matches = len(p.findall(clean))
                if matches > 0: 
                    term_counts[t] += matches
        except: 
            continue

    # PROCESAR DATOS
    df = pd.DataFrame(list(term_counts.items()), columns=['Termino', 'Frecuencia'])
    df = df[df['Frecuencia'] > 0].sort_values('Frecuencia', ascending=False).head(20)
    
    if df.empty: 
        return print("❌ Sin datos suficientes para graficar.")

    # COLORES
    lista_colores = []
    roots_en_grafica = set() 
    for term in df['Termino']:
        padre = term_to_root.get(term.lower(), term.lower())
        color = colores_tfg.get(padre, '#999999')
        lista_colores.append(color)
        roots_en_grafica.add(padre)

    # =========================================================================
    # CONFIGURACIÓN DE GRÁFICA DE ALTA VISIBILIDAD (ESPECIAL POWERPOINT/PDF)
    # =========================================================================
    plt.figure(figsize=(16, 10), facecolor='white') # Altura subida a 10 para dar más espacio vertical
    ax = plt.gca()
    
    plot = sns.barplot(
        data=df, x='Frecuencia', y='Termino', 
        palette=lista_colores, hue='Termino',
        legend=False, edgecolor='black', linewidth=1.5, alpha=0.85
    )

    patches = [mpatches.Patch(color=colores_tfg.get(r, '#999'), label=r.upper()) for r in sorted(list(roots_en_grafica))]
    
    # Leyenda grande y limpia
    leg = plt.legend(
        handles=patches, 
        title="Categoría Raíz (Semántica)", 
        bbox_to_anchor=(0.5, 1.14),  
        loc='upper center', 
        ncol=min(len(patches), 4), 
        frameon=True, fontsize=12, title_fontsize=13, shadow=True # Subidos tamaños de leyenda
    )
    plt.setp(leg.get_title(), fontweight='bold')
    
    # Título grande
    plt.title('Análisis de Frecuencia: Términos Detectados por Ontología', 
              fontsize=18, fontweight='bold', pad=70, color='#333') # Subido a 18 y pad a 70
    
    # ETIQUETAS AL FINAL DE LAS BARRAS (¡Letras grandes!)
    max_val = df['Frecuencia'].max()
    for p in plot.patches:
        width = p.get_width()
        if width > 0:
            # Desplazamos un pelín más el texto a la derecha para que la fuente grande no pise la barra
            plt.text(width + (max_val * 0.015), p.get_y() + p.get_height()/2, 
                     f'{int(width):,}', va='center', fontsize=12, fontweight='bold', color='#222') # Subido a 12

    # FORMATO DE LOS EJES (Texto lateral y números inferiores grandes)
    for i, label in enumerate(ax.get_yticklabels()):
        label.set_fontweight('bold')
        label.set_fontsize(14) # Nombres de términos grandes para que se lean bien
        label.set_color(lista_colores[i])

    plt.xticks(fontsize=13, fontweight='bold') # Números del eje X grandes
    plt.xlabel('Número total de menciones', fontsize=14, fontweight='bold', labelpad=15) # Subido a 14
    plt.ylabel('') 
    
    plt.grid(axis='x', linestyle='--', alpha=0.3)
    sns.despine()
    
    plt.subplots_adjust(top=0.76) # Ajustado el margen superior para que no se solape la leyenda gigante

    # GUARDADO VECTORIAL
    ruta_dir = 'Graficas'
    os.makedirs(ruta_dir, exist_ok=True)
    plt.savefig(os.path.join(ruta_dir, 'frecuencia_ontologia_final.svg'), bbox_inches='tight')
    plt.show()

    # TABLA
    print("\n📊 DETALLE DE FRECUENCIAS DETECTADAS:")
    df_tabla = df.copy()
    total_m = df_tabla['Frecuencia'].sum()
    df_tabla['%'] = (df_tabla['Frecuencia'] / total_m * 100).round(2)
    df_tabla.columns = ['Nombre', 'Menciones', 'Porcentaje (%)']
    
    display(HTML("""
        <style>
            .itables table { 
                margin-left: 0 !important; 
                margin-right: auto !important; 
                width: 700px !important; 
                font-size: 0.85em !important;
                border-collapse: collapse !important;
            }
            .itables th { background-color: #f7fafc !important; border-bottom: 2px solid #edf2f7 !important; text-align: left !important;}
            .itables td { border-bottom: 1px solid #edf2f7 !important; text-align: left !important;}
        </style>
    """))
    show(df_tabla, 
         paging=False, 
         search=False, 
         info=False, 
         dom='t',
         columnDefs=[
             {"width": "250px", "targets": 0}, 
             {"width": "150px", "targets": 1}, 
             {"width": "150px", "targets": 2}, 
             {"className": "dt-left", "targets": "_all"}
         ])

    # GUARDADO CSV
    csv_data = df_tabla.to_csv(index=False)
    b64_csv = base64.b64encode(csv_data.encode()).decode()
    payload = f"data:text/csv;base64,{b64_csv}"
    
    display(HTML(f"""
    <div style="text-align: left; margin-top: 15px;">
        <span style="font-size: 10px; color: #718096; display: block; margin-bottom: 8px;">📁 Guardado en: {ruta_dir}/frecuencia_ontologia_final.svg</span>
        <a href="{payload}" download="frecuencia_ontologia.csv" style="
            display: inline-block; background-color: #4A5568; color: white;
            padding: 6px 14px; text-decoration: none; border-radius: 6px;
            font-weight: 600; font-family: sans-serif; font-size: 12px;
        ">📥 Descargar Frecuencias CSV</a>
    </div>
    """))

# EJECUTAR CODIGO
run_analysis_colored_fix(limit=300)

<div class="seccion-ocultar">

### Descarga de HTML seguro

</div>

<div class="seccion-ocultar">

En esta gráfica se ven diveros documentos de los cuales sepueden ver el título, los terminos que se encuentran en esa página y con que frecuencia de menciones y finalmente se puede descargar el HTML.

</div>

In [ ]:
#OCULTAR
def show_html_terms_frecuency(neo_driver, mongo_uri, db_name, limit_docs=50):
    # CONEXIÓN MONGODB
    client = MongoClient(mongo_uri)
    db = client[db_name]
    fs = gridfs.GridFS(db)

    # CONSULTA
    vocab_query = """
    MATCH (t:Term) 
    OPTIONAL MATCH (s:Synonym)-[:IS_SYNONYM_OF]->(t) 
    RETURN t.name AS root, collect(s.name) AS synonyms
    """
    pages_query = """
    MATCH (p:Page) 
    WHERE p.text IS NOT NULL 
    RETURN 
        coalesce(p.title, p.url) AS nombre_sitio, 
        p.text AS text, 
        p.url AS url_original
    LIMIT $limit
    """
    
    try:
        with neo_driver.session() as session:
            vocab_data = session.run(vocab_query).data()
            pages_data = session.run(pages_query, limit=limit_docs).data()

        reporte_data = []
        carpeta_destino = 'Descargas_HTML'
        os.makedirs(carpeta_destino, exist_ok=True)
        print(f"📂 Carpeta de destino preparada: {os.path.abspath(carpeta_destino)}")

        for page in pages_data:
            nombre_sitio = page['nombre_sitio']
            texto_completo = page['text']
            url_original = page['url_original']
            
            categorias_list = []
            detalles_html = []
            total_matches = 0
            
            for item in vocab_data:
                root = item['root']
                
                lista_sucia = [root] + item['synonyms']
                variantes = list(set(lista_sucia)) 

                hallazgos = []
                suma_root = 0
                
                for v in variantes:
                    if not v: continue 
                    matches = re.findall(rf'\b{re.escape(v)}\b', texto_completo, re.IGNORECASE)
                    if matches:
                        hallazgos.append(f"{v} ({len(matches)})")
                        suma_root += len(matches)
                
                if suma_root > 0:
                    categorias_list.append(root.upper())
                    detalles_html.append(f"<b>{root.upper()}:</b> {', '.join(hallazgos)}")
                    total_matches += suma_root

            if total_matches > 0:
                regex_url = f"^{re.escape(url_original.strip())}/?$"
                archivo_mongo = db['fs.files'].find_one({"metadata.source_url": {"$regex": regex_url}})
                
                if archivo_mongo:
                    try:
                        contenido_html = fs.get(archivo_mongo['_id']).read()
                        
                        safe_name = re.sub(r'[\\/*?:"<>|]', "_", str(nombre_sitio))[:50]
                        filename = f"evidencia_{safe_name}.html"
                        
                        ruta_completa = os.path.join(carpeta_destino, filename)
                        with open(ruta_completa, 'wb') as f:
                            f.write(contenido_html)
                        b64_html = base64.b64encode(contenido_html).decode('utf-8')
                        data_uri = f"data:text/html;base64,{b64_html}"
                        
                        btn_html = f'''
                        <a href="{data_uri}" download="{filename}" style="text-decoration:none;">
                            <div style="background-color:#2F855A; color:white; padding:8px 12px; border-radius:4px; 
                            text-align:center; font-weight:bold; font-size:12px; width:100px;">📥 Descargar HTML</div>
                        </a>
                        <div style="font-size:10px; color:gray; margin-top:5px;">✅ Guardado en disco</div>'''
                        
                    except Exception as e_save:
                        btn_html = f"<span style='color:red'>Error al guardar: {e_save}</span>"
                else:
                    btn_html = "<span style='color:gray'>No en GridFS</span>"

                reporte_data.append({
                    "Sitio Web (Título)": f"<div style='font-weight:bold; color:#333;'>{nombre_sitio}</div>",
                    "Categorías": " | ".join(categorias_list),
                    "Frecuencia Total": f"<span style='font-size:14px; font-weight:bold;'>{total_matches}</span>",
                    "Desglose Detallado": f"<div style='line-height:1.6; font-size:12px;'>{'<br>'.join(detalles_html)}</div>",
                    "Acción": btn_html
                })

        df = pd.DataFrame(reporte_data)
        
        if not df.empty:
            estilos = [
                {'selector': 'table', 'props': [('border-collapse', 'collapse'), ('width', '100%')]},
                {'selector': 'th', 'props': [('background-color', '#f8f9fa'), ('color', '#333'), ('text-align', 'left')]},
                {'selector': '.col0', 'props': [('width', '20%')]}, 
                {'selector': '.col1', 'props': [('width', '15%')]}, 
                {'selector': '.col2', 'props': [('width', '10%'), ('text-align', 'center')]}, 
                {'selector': '.col3', 'props': [('width', '40%')]}, 
                {'selector': '.col4', 'props': [('width', '15%')]}  
            ]
            
            html_final = df.style.set_table_styles(estilos).hide(axis='index').to_html()
            display(HTML(html_final))
            print(f"✅ Auditoría completada. Los archivos HTML se han guardado en la carpeta '{carpeta_destino}'.")
        else:
            print("⚠️ No se encontraron coincidencias en la muestra seleccionada.")
        
        client.close()

    except Exception as e:
        print(f"❌ Error crítico: {e}")
        import traceback
        traceback.print_exc()

# EJECUTAR
show_html_terms_frecuency(neo_driver, MONGO_URI, MONGO_DB_NAME, limit_docs=30)

<div class="seccion-ocultar">

### Descarga HTML seguro con filtro

</div>

<div class="seccion-ocultar">

Con este codigo, se nos pide que indiquemos via texto un término de los que son raiz, para poder ver 10 html de ese aleatorios y poder ver que es lo que tienen en sus páginas pero de manera segura.  
Así se da la opción de descargar el HTML para poder análizarlo en cualquier otro momento.

</div>

In [ ]:
#OCULTAR

def download_html_by_category(categoria):
    try:
        # CONEXIÓN
        client = MongoClient(MONGO_URI)
        db = client[MONGO_DB_NAME]
        fs = gridfs.GridFS(db)
        urls = []

        # CONSULTA
        if neo_driver:
            with neo_driver.session() as session:
                query_neo = """
                MATCH (root:Term {name: $cat})
                OPTIONAL MATCH (synonym:Synonym)-[:IS_SYNONYM_OF]->(root)
                WITH root, collect(synonym.name) + root.name AS all_keywords
                MATCH (p:Page)
                WHERE ANY(word IN all_keywords WHERE toLower(p.text) CONTAINS toLower(word))
                RETURN p.url AS url
                LIMIT 10
                """
                results = session.run(query_neo, cat=categoria)
                urls = [record["url"] for record in results]
        
        print(f"🔎 Neo4j ha devuelto {len(urls)} URLs para '{categoria}'.")

        if not urls:
            print("❌ No se encontraron URLs en Neo4j para esta temática.")
            return
        carpeta_destino = 'Descargas_HTML/busqueda'
        os.makedirs(carpeta_destino, exist_ok=True)
        print(f"📂 Los archivos se guardarán en: {os.path.abspath(carpeta_destino)}\n")

        # RESULTADOS
        html_output = f"""
        <div style="font-family: Arial, sans-serif; max-width: 800px;">
            <h2 style='color: #2F855A; border-bottom: 2px solid #2F855A;'>📁 Evidencias encontradas: {categoria}</h2>
            <p style='color: #718096;'>Los archivos se han guardado en la carpeta <b>{carpeta_destino}</b>.</p>
        </div>
        """
        
        encontrados = 0
        for i, url in enumerate(urls, 1):
            url_limpia = url.strip()
            regex_url = f"^{url_limpia}/?$"
            
            archivo = db['fs.files'].find_one({
                "metadata.source_url": {"$regex": regex_url}
            })
            
            if archivo:
                encontrados += 1
                f_id = archivo['_id']
                nombre_archivo = f"evidencia_{categoria}_{i}.html"

                contenido_binario = fs.get(f_id).read()
                
                ruta_completa = os.path.join(carpeta_destino, nombre_archivo)
                try:
                    with open(ruta_completa, 'wb') as f:
                        f.write(contenido_binario)
                    msg_guardado = f"✅ Guardado localmente"
                except Exception as e_save:
                    msg_guardado = f"❌ Error al guardar: {e_save}"
 
                b64_html = base64.b64encode(contenido_binario).decode('utf-8')
                data_uri = f"data:text/html;base64,{b64_html}"
                
                html_output += f"""
                <div style='margin-bottom: 15px; border: 1px solid #e2e8f0; padding: 15px; border-radius: 8px; background: #f8fafc; box-shadow: 2px 2px 5px rgba(0,0,0,0.05);'>
                    <div style='margin-bottom: 8px;'>
                        <b style='color: #2d3748; font-size: 14px;'>🌐 URL:</b> 
                        <code style='color: #c53030; background: #fff5f5; padding: 2px 4px;'>{url_limpia}</code>
                    </div>
                    
                    <a href="{data_uri}" download="{nombre_archivo}" style="
                        display: inline-block;
                        background-color: #2F855A;
                        color: white;
                        padding: 10px 18px;
                        text-decoration: none;
                        border-radius: 6px;
                        font-weight: bold;
                        font-size: 13px;
                        transition: background 0.3s;
                    " onmouseover="this.style.backgroundColor='#276749'" onmouseout="this.style.backgroundColor='#2F855A'">
                        📥 Descargar HTML Completo
                    </a>
                    <span style='margin-left: 15px; font-size: 11px; color: green;'>{msg_guardado} en: {nombre_archivo}</span>
                </div>
                """
            else:
                pass

        if encontrados > 0:
            display(HTML(html_output))
            print(f"🎉 Proceso finalizado. {encontrados} archivos guardados correctamente.")
        else:
            print("⚠️ No se encontraron archivos físicos en MongoDB para las URLs proporcionadas por Neo4j.")
            
        client.close()
    except Exception as e:
        print(f"❌ Error crítico: {e}")

# EJECUTAR
print("--- BUSCADOR DE EVIDENCIAS DARK WEB (GUARDADO AUTOMÁTICO) ---")
seleccion = input("Escribe la categoría (ej. drugs, hacking, marketplace): ").strip()

if seleccion:
    print(f"⏳ Procesando archivos de '{seleccion}'... por favor espera.")
    download_html_by_category(seleccion)
else:
    print("Debes introducir una temática.")

### Descarga HTML seguro y representación Neo4J

Con este codigo, primero indica que se pase por texto el primer término de los 2 que se buscarán, y luego lo mismo con el segundo término.  
Una vez tiene los dos terminos, imprime el grafo de Neo4J que se encuentra al buscar ambos terminos y, finalmente indoca los 10 enlaces de cada uno sanitizados para que se pueda ver las páginas que hay en ese grafo Neo y se pueden descargar en HTML.

In [ ]:
# VER LAS TEMATICAS
def neo4j_terms_comparison():
    try:
        with neo_driver.session() as session:
            res = session.run("MATCH (t:Term) RETURN t.name as name ORDER BY name")
            terminos = [r["name"] for r in res]
        
        badges = "".join([f'<span style="background:#ebf8ff; color:#2b6cb0; padding:4px 12px; border-radius:15px; font-size:12px; font-weight:bold; border:1px solid #bee3f8; margin:3px; display:inline-block;">{t}</span>' for t in terminos])
        display(HTML(f"""
        <div style="background:#f7fafc; padding:20px; border-radius:12px; border:1px solid #e2e8f0; margin-bottom:20px; font-family: sans-serif;">
            <h4 style="margin:0 0 10px 0; color:#2d3748;">🔍 Temáticas disponibles en la Ontología:</h4>
            <div>{badges}</div>
        </div>
        """))
        return terminos
    except Exception as e:
        print(f"Nota: No se pudo conectar con Neo4j para listar términos ({e})")
        return []

# GRAFO 
def visualize_graph_save(cat_a, cat_b):
    try:
        from pyvis.network import Network
        net = Network(height="600px", width="100%", bgcolor="#ffffff", font_color="black", notebook=True, directed=True, cdn_resources='remote')

        net.set_options("""
        var options = {
          "physics": {
            "forceAtlas2Based": {"gravitationalConstant": -150, "centralGravity": 0.01, "springLength": 200, "springConstant": 0.05},
            "solver": "forceAtlas2Based"
          }
        }
        """)
        # CONSULTA
        query = """
        CALL {
            MATCH (t1:Term) WHERE toLower(t1.name) = toLower($catA)
            OPTIONAL MATCH (s1:Synonym)-[:IS_SYNONYM_OF]->(t1)
            WITH t1, collect(toLower(s1.name)) + [toLower(t1.name)] AS keys1
            MATCH (p1:Page)
            WHERE p1.text IS NOT NULL AND ANY(word IN keys1 WHERE toLower(p1.text) CONTAINS word)
            RETURN t1.name AS term_name, elementId(t1) AS t_id, p1 AS p, elementId(p1) AS p_id
            LIMIT 10
        }
        RETURN term_name, t_id, p, p_id
        UNION ALL
        CALL {
            MATCH (t2:Term) WHERE toLower(t2.name) = toLower($catB)
            OPTIONAL MATCH (s2:Synonym)-[:IS_SYNONYM_OF]->(t2)
            WITH t2, collect(toLower(s2.name)) + [toLower(t2.name)] AS keys2
            MATCH (p2:Page)
            WHERE p2.text IS NOT NULL AND ANY(word IN keys2 WHERE toLower(p2.text) CONTAINS word)
            RETURN t2.name AS term_name, elementId(t2) AS t_id, p2 AS p, elementId(p2) AS p_id
            LIMIT 10
        }
        RETURN term_name, t_id, p, p_id
        """
        
        with neo_driver.session() as session:
            results = session.run(query, catA=cat_a, catB=cat_b)
            nodos_creados = set()
            
            for record in results:
                t_name = record['term_name']
                t_id = record['t_id']
                p_id = record['p_id']
                p = record['p']
                
                if t_id not in nodos_creados:
                    color = "#e53e3e" if t_name.lower() == cat_a.lower() else "#38a169"
                    net.add_node(t_id, label=t_name.upper(), color=color, size=40, shape="diamond")
                    nodos_creados.add(t_id)
                
                if p_id not in nodos_creados:
                    titulo = p.get('title', 'Web')
                    label_p = (str(titulo)[:25] + "...") if titulo else p['url'][:20]
                    net.add_node(p_id, label=label_p, title=f"URL: {p['url']}", color="#3182ce", size=20)
                    nodos_creados.add(p_id)
                
                net.add_edge(p_id, t_id, color="#cbd5e0", width=1.5)

        os.makedirs('Graficas', exist_ok=True)
        path_html = os.path.join('Graficas', 'grafo_dual_comparativa.html')
        net.show(path_html)
        
        display(HTML(f"<h3 style='color: #2D3748; font-family: sans-serif; border-left: 5px solid #2b6cb0; padding-left: 10px;'>🔗 Comparativa Estructural: {cat_a.upper()} vs {cat_b.upper()}</h3>"))
        display(IFrame(path_html, width="100%", height="600px"))
            
    except Exception as e:
        print(f"❌ Error en Grafo: {e}")

# DESCARGAR HTML
def download_both_evidence_save(cat_a, cat_b):
    try:
        client = MongoClient(MONGO_URI)
        db = client[MONGO_DB_NAME]
        fs = gridfs.GridFS(db)
        
        carpeta_destino = 'Descargas_HTML/comparacion_neo4j'
        os.makedirs(carpeta_destino, exist_ok=True)

        query_urls = """
        CALL {
            MATCH (t:Term) WHERE toLower(t.name) = toLower($catA)
            OPTIONAL MATCH (s:Synonym)-[:IS_SYNONYM_OF]->(t)
            WITH t, collect(toLower(s.name)) + [toLower(t.name)] AS keys
            MATCH (p:Page)
            WHERE p.text IS NOT NULL AND ANY(word IN keys WHERE toLower(p.text) CONTAINS word)
            RETURN p.url AS url, p.title AS title, t.name AS categoria LIMIT 10
        }
        RETURN url, title, categoria
        UNION ALL
        CALL {
            MATCH (t:Term) WHERE toLower(t.name) = toLower($catB)
            OPTIONAL MATCH (s:Synonym)-[:IS_SYNONYM_OF]->(t)
            WITH t, collect(toLower(s.name)) + [toLower(t.name)] AS keys
            MATCH (p:Page)
            WHERE p.text IS NOT NULL AND ANY(word IN keys WHERE toLower(p.text) CONTAINS word)
            RETURN p.url AS url, p.title AS title, t.name AS categoria LIMIT 10
        }
        RETURN url, title, categoria
        """
        
        with neo_driver.session() as session:
            res = session.run(query_urls, catA=cat_a, catB=cat_b)
            paginas = list(res)

        html_out = f"<h4 style='font-family:sans-serif; color: #2F855A; margin-top: 25px;'>📥 Muestras de Evidencias Guardadas:</h4>"
        
        count_saved = 0
        for i, pag in enumerate(paginas, 1):
            url_l = pag['url'].strip()
            categoria = pag['categoria']
            archivo = db['fs.files'].find_one({"metadata.source_url": {"$regex": f"^{re.escape(url_l)}/?$"}})
            
            if archivo:
                contenido = fs.get(archivo['_id']).read()
                safe_cat = categoria.replace(" ", "_")
                filename = f"evidencia_{safe_cat}_{i}.html"
                ruta_completa = os.path.join(carpeta_destino, filename)
                
                with open(ruta_completa, 'wb') as f:
                    f.write(contenido)
                count_saved += 1
                
                b64 = base64.b64encode(contenido).decode('utf-8')
                badge_col = "#e53e3e" if categoria.lower() == cat_a.lower() else "#38a169"
                
                html_out += f"""
                <div style='margin-bottom: 8px; border-left: 5px solid {badge_col}; padding: 10px; background: #fff; border-radius: 4px; font-family:sans-serif; box-shadow: 0 1px 2px rgba(0,0,0,0.1);'>
                    <span style='background:{badge_col}; color:white; padding:2px 6px; border-radius:3px; font-size:10px;'>{categoria.upper()}</span>
                    <div style='font-size: 13px; margin: 5px 0;'><b>{pag['title'] if pag['title'] else 'Documento HTML'}</b></div>
                    <a href="data:text/html;base64,{b64}" download="{filename}" style="text-decoration:none; font-size:11px; color:#3182ce;">📥 Descargar copia local</a>
                </div>"""
        
        display(HTML(html_out))
        print(f"🎉 Éxito: {count_saved} archivos HTML guardados en '{carpeta_destino}'.")
        client.close()
    except Exception as e:
        print(f"❌ Error MongoDB: {e}")

# EJECUTAR
clear_output()
terminos_disponibles = neo4j_terms_comparison()

print("--- CONFIGURACIÓN DEL ANÁLISIS ---")
tema_1 = input("👉 Introduce el primer término (ej: drugs): ").strip().lower()
tema_2 = input("👉 Introduce el segundo término (ej: hacking): ").strip().lower()

if tema_1 and tema_2:
    if tema_1 == tema_2:
        print("⚠️ Debes introducir dos temas distintos.")
    else:
        print(f"\n⌛ Generando comparativa para '{tema_1}' y '{tema_2}'...")
        # GRAFO
        visualize_graph_save(tema_1, tema_2)
        # HTML
        download_both_evidence_save(tema_1, tema_2)
else:
    print("❌ Error: Ambos términos son obligatorios.")

### Análisis de matríz de correlación

Se realiza una matriz de correlación únicamente del triangulo superior con una correlación que sea superior al 0.1 gracias al Coeficiente de Pearson.  
Además se imprime un top 5 de las relaciones más fuertes que se encuentran en la matríz.  
Para asegurar que sean páginas únicas se pone DISTINCT.

In [ ]:
warnings.filterwarnings("ignore")
def generate_full_correlation_report(neo_driver, threshold=0.1, n_top=5):
    if not neo_driver:
        print("❌ Error: El driver de Neo4j no está disponible.")
        return None

    # CONSULTA
    query = """
    MATCH (t:Term)<-[:IS_SYNONYM_OF]-(s:Synonym)<-[:MENTIONS]-(p:Page)
    RETURN p.url AS doc_id, t.name AS term_name
    """
    
    try:
        with neo_driver.session() as session:
            records = session.run(query).data()
            if not records:
                print("⚠️ No se encontraron datos para correlacionar.")
                return None
            df_raw = pd.DataFrame(records)

        # CÁLCULO DE COMPOSICIÓN PORCENTUAL
        mentions_matrix = pd.crosstab(df_raw['doc_id'], df_raw['term_name']).rename_axis(None, axis=1)
        total_menciones_por_pagina = mentions_matrix.sum(axis=1)
        relative_perc_matrix = (mentions_matrix.div(total_menciones_por_pagina, axis=0) * 100).round(2)

        # MATRIZ DE CORRELACIÓN
        correlation_matrix = relative_perc_matrix.corr()

        # GRAFICA 
        mask_tri = np.tril(np.ones_like(correlation_matrix, dtype=bool), k=0)
        mask_threshold = correlation_matrix < threshold
        final_mask = mask_tri | mask_threshold

        plt.figure(figsize=(14, 12))
        sns.set_theme(style="white")
        cmap_custom = sns.diverging_palette(10, 130, as_cmap=True)

        ax = sns.heatmap(
            correlation_matrix, 
            mask=final_mask,
            annot=True, 
            fmt=".2f", 
            cmap=cmap_custom,
            center=0, vmin=-1, vmax=1,
            square=True, linewidths=.5,
            annot_kws={"size": 12, "weight": "bold"},
            cbar_kws={"shrink": .7, "label": "Coeficiente de Pearson (r)"}
        )
        # Modificar el tamaño de la etiqueta de la barra de color
        ax.figure.axes[-1].yaxis.label.set_size(13)
        ax.figure.axes[-1].yaxis.label.set_weight('bold')
        ax.figure.axes[-1].tick_params(labelsize=11)

        ax.xaxis.tick_top()
        ax.xaxis.set_label_position('top') 
        # 3. TÍTULO GRANDE Y DESTACADO
        plt.title('Matriz de Correlación (Pearson)', fontsize=18, fontweight='bold', pad=35)
        
        # 4. LETRAS DE LOS EJES MÁS GRANDES
        plt.xticks(rotation=45, ha='left', fontsize=12, fontweight='bold')
        plt.yticks(fontsize=12, fontweight='bold')
        
        plt.tight_layout()
        carpeta_destino = r"C:\Users\sandr\TFG\Analisis\Graficas"
        os.makedirs(carpeta_destino, exist_ok=True)
        print(f"📂 Los archivos se guardarán en: {os.path.abspath(carpeta_destino)}\n")
        
        # Guardar la imagen en alta resolución (calidad imprenta 300 dpi)
        path_imagen_tfg = os.path.join(carpeta_destino, "matriz_correlacion_pearson.svg")
        plt.savefig(path_imagen_tfg, dpi=300, bbox_inches='tight')
        print(f"🖼️ Gráfica guardada con éxito en: {path_imagen_tfg}")
        plt.show()

        # TOP 5 RELACIONES
        upper_tri = correlation_matrix.where(np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool))
        pairs = upper_tri.unstack().dropna()
        
        if pairs.empty:
            print("No hay correlaciones suficientes.")
            return correlation_matrix

        top_df = pairs.sort_values(ascending=False).head(n_top).reset_index()
        top_df.columns = ['Término B', 'Término A', 'Coef. Pearson']
        
        def interpretar(val):
            if val > 0.7: return "Muy Fuerte"
            if val > 0.5: return "Fuerte"
            if val > 0.3: return "Moderada"
            return "Débil"
        
        top_df['Fuerza'] = top_df['Coef. Pearson'].apply(interpretar)
        top_df['Asociación'] = top_df['Término A'].str.upper() + " + " + top_df['Término B'].str.upper()
        df_tabla_top = top_df[['Asociación', 'Coef. Pearson', 'Fuerza']].copy()

        print(f"\n📊 TOP {n_top} ASOCIACIONES TEMÁTICAS MÁS FUERTES:")
        display(HTML("""
            <style>
                .itables table { 
                    margin-left: 0 !important; 
                    margin-right: auto !important; 
                    width: 700px !important; 
                    max-width: 800px !important;
                    font-size: 0.85em !important;
                    border-collapse: collapse !important;
                    table-layout: fixed !important; 
                }
                .itables th, .itables td { 
                    padding: 10px 15px !important; 
                    white-space: nowrap !important;
                    overflow: hidden;
                    text-overflow: ellipsis;
                }
                .itables th { background-color: #f7fafc !important; border-bottom: 2px solid #edf2f7 !important; }
                .itables td { border-bottom: 1px solid #edf2f7 !important; }
            </style>
            """))
        show(df_tabla_top, 
             paging=False, 
             search=False, 
             info=False, 
             dom='t',
             autoWidth=False, 
             columnDefs=[
                 {"width": "350px", "targets": 0}, # Asociación (más ancho para dos palabras)
                 {"width": "175px", "targets": 1}, # Coef. Pearson
                 {"width": "175px", "targets": 2}, # Fuerza
                 {"className": "dt-left", "targets": "_all"}
             ])

        # GUARDADO
        ruta_dir = 'Graficas'
        os.makedirs(ruta_dir, exist_ok=True)
        path_completo = os.path.join(ruta_dir, "top_asociaciones_pearson.csv")
        df_tabla_top.to_csv(path_completo, index=False)

        csv_data = df_tabla_top.to_csv(index=False)
        b64_csv = base64.b64encode(csv_data.encode()).decode()
        payload = f"data:text/csv;base64,{b64_csv}"
        
        display(HTML(f"""
        <div style="text-align: left; margin-top: 20px;">
            <p style="font-size: 11px; color: #718096; margin-bottom: 5px;">✅ Top asociaciones guardado en: {path_completo}</p>
            <a href="{payload}" download="top_asociaciones.csv" style="
                display: inline-block; background-color: #3182ce; color: white;
                padding: 8px 16px; text-decoration: none; border-radius: 6px;
                font-weight: bold; font-family: sans-serif; font-size: 12px;
            ">📥 Descargar CSV Top Asociaciones</a>
        </div>
        """))

        return correlation_matrix

    except Exception as e:
        print(f"❌ Error crítico: {e}")
        return None
        
# EJECUTAR
ejecutar = generate_full_correlation_report(neo_driver, threshold=0.1, n_top=5)

In [ ]:
warnings.filterwarnings("ignore")
def generate_full_correlation_report(neo_driver, threshold=0.1, n_top=5):
    if not neo_driver:
        print("❌ Error: El driver de Neo4j no está disponible.")
        return None

    # CONSULTA
    query = """
    MATCH (t:Term)<-[:IS_SYNONYM_OF]-(s:Synonym)<-[:MENTIONS]-(p:Page)
    RETURN p.url AS doc_id, t.name AS term_name
    """
    
    try:
        with neo_driver.session() as session:
            records = session.run(query).data()
            if not records:
                print("⚠️ No se encontraron datos para correlacionar.")
                return None
            df_raw = pd.DataFrame(records)

        # CÁLCULO DE COMPOSICIÓN PORCENTUAL
        mentions_matrix = pd.crosstab(df_raw['doc_id'], df_raw['term_name']).rename_axis(None, axis=1)
        total_menciones_por_pagina = mentions_matrix.sum(axis=1)
        relative_perc_matrix = (mentions_matrix.div(total_menciones_por_pagina, axis=0) * 100).round(2)

        # MATRIZ DE CORRELACIÓN
        correlation_matrix = relative_perc_matrix.corr()

        # =========================================================================
        # CONFIGURACIÓN DE RECORTE Y MÁSCARAS
        # =========================================================================
        mask_tri = np.tril(np.ones_like(correlation_matrix, dtype=bool), k=0)
        mask_threshold = correlation_matrix < threshold
        final_mask = mask_tri | mask_threshold

        # =========================================================================
        # REAJUSTE DE TAMAÑO GIGANTE (ESPECIAL POWERPOINT/PDF)
        # =========================================================================
        plt.figure(figsize=(16, 14), facecolor='white') # Lienzo más grande para albergar fuentes grandes
        sns.set_theme(style="white")
        cmap_custom = sns.diverging_palette(10, 130, as_cmap=True)

        ax = sns.heatmap(
            correlation_matrix, 
            mask=final_mask,
            annot=True, 
            fmt=".2f", 
            cmap=cmap_custom,
            center=0, vmin=-1, vmax=1,
            square=True, linewidths=.8, # Un pelín más de separación entre celdas
            annot_kws={"size": 14, "weight": "bold"}, # ¡Números dentro de la matriz subidos a 14pt!
            cbar_kws={"shrink": .7, "label": "Coeficiente de Pearson (r)"}
        )
        
        # CONFIGURACIÓN DE LA BARRA DE COLOR LATERIAL (Leyenda de color grande)
        cbar_ax = ax.figure.axes[-1]
        cbar_ax.yaxis.label.set_size(15) # Texto de la barra a 15pt
        cbar_ax.yaxis.label.set_weight('bold')
        cbar_ax.tick_params(labelsize=13, labelfontweight='bold') # Números de la barra a 13pt

        ax.xaxis.tick_top()
        ax.xaxis.set_label_position('top') 
        
        # TÍTULO GIGANTE
        plt.title('Matriz de Correlación (Pearson)', fontsize=20, fontweight='bold', pad=45)
        
        # TEXTO DE LOS EJES (Nombres de los términos en grande)
        plt.xticks(rotation=45, ha='left', fontsize=14, fontweight='bold') # Eje X a 14pt
        plt.yticks(fontsize=14, fontweight='bold') # Eje Y a 14pt
        
        plt.tight_layout()
        carpeta_destino = r"C:\Users\sandr\TFG\Analisis\Graficas"
        os.makedirs(carpeta_destino, exist_ok=True)
        print(f"📂 Los archivos se guardarán en: {os.path.abspath(carpeta_destino)}\n")
        
        # Guardado nativo en formato vectorial .SVG
        path_imagen_tfg = os.path.join(carpeta_destino, "matriz_correlacion_pearson.svg")
        plt.savefig(path_imagen_tfg, bbox_inches='tight')
        print(f"🖼️ Gráfica guardada con éxito en: {path_imagen_tfg}")
        plt.show()

        # TOP 5 RELACIONES
        upper_tri = correlation_matrix.where(np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool))
        pairs = upper_tri.unstack().dropna()
        
        if pairs.empty:
            print("No hay correlaciones suficientes.")
            return correlation_matrix

        top_df = pairs.sort_values(ascending=False).head(n_top).reset_index()
        top_df.columns = ['Término B', 'Término A', 'Coef. Pearson']
        
        def interpretar(val):
            if val > 0.7: return "Muy Fuerte"
            if val > 0.5: return "Fuerte"
            if val > 0.3: return "Moderada"
            return "Débil"
        
        top_df['Fuerza'] = top_df['Coef. Pearson'].apply(interpretar)
        top_df['Asociación'] = top_df['Término A'].str.upper() + " + " + top_df['Término B'].str.upper()
        df_tabla_top = top_df[['Asociación', 'Coef. Pearson', 'Fuerza']].copy()

        print(f"\n📊 TOP {n_top} ASOCIACIONES TEMÁTICAS MÁS FUERTES:")
        display(HTML("""
            <style>
                .itables table { 
                    margin-left: 0 !important; 
                    margin-right: auto !important; 
                    width: 700px !important; 
                    max-width: 800px !important;
                    font-size: 0.85em !important;
                    border-collapse: collapse !important;
                    table-layout: fixed !important; 
                }
                .itables th, .itables td { 
                    padding: 10px 15px !important; 
                    white-space: nowrap !important;
                    overflow: hidden;
                    text-overflow: ellipsis;
                }
                .itables th { background-color: #f7fafc !important; border-bottom: 2px solid #edf2f7 !important; }
                .itables td { border-bottom: 1px solid #edf2f7 !important; }
            </style>
            """))
        show(df_tabla_top, 
             paging=False, 
             search=False, 
             info=False, 
             dom='t',
             autoWidth=False, 
             columnDefs=[
                 {"width": "350px", "targets": 0}, 
                 {"width": "175px", "targets": 1}, 
                 {"width": "175px", "targets": 2}, 
                 {"className": "dt-left", "targets": "_all"}
             ])

        # GUARDADO CSV
        ruta_dir = 'Graficas'
        os.makedirs(ruta_dir, exist_ok=True)
        path_completo = os.path.join(ruta_dir, "top_asociaciones_pearson.csv")
        df_tabla_top.to_csv(path_completo, index=False)

        csv_data = df_tabla_top.to_csv(index=False)
        b64_csv = base64.b64encode(csv_data.encode()).decode()
        payload = f"data:text/csv;base64,{b64_csv}"
        
        display(HTML(f"""
        <div style="text-align: left; margin-top: 20px;">
            <p style="font-size: 11px; color: #718096; margin-bottom: 5px;">✅ Top asociaciones guardado en: {path_completo}</p>
            <a href="{payload}" download="top_asociaciones.csv" style="
                display: inline-block; background-color: #3182ce; color: white;
                padding: 8px 16px; text-decoration: none; border-radius: 6px;
                font-weight: bold; font-family: sans-serif; font-size: 12px;
            ">📥 Descargar CSV Top Asociaciones</a>
        </div>
        """))

        return correlation_matrix

    except Exception as e:
        print(f"❌ Error crítico: {e}")
        return None
        
# EJECUTAR
ejecutar = generate_full_correlation_report(neo_driver, threshold=0.1, n_top=5)

A partir del código anterior, se realiza una matriz con gráficas de dispersión para comparar cada término con los otros que tenemos en base a su frecuencia y con un límite de 3000 páginas para no saturar.  
La última gráfica de cada termino es la densidad, es decir, en las páginas si hay más con poco del término o si hay más páginas que tienen mucha información de ese término.

In [ ]:
def generate_auto_intensity_matrix_porcentual(neo_driver, limit_pages=3000):
    # CONSULTA
    vocab_query = """
    MATCH (t:Term)
    OPTIONAL MATCH (s:Synonym)-[:IS_SYNONYM_OF]->(t)
    RETURN t.name AS root, collect(s.name) + t.name AS words
    """
    vocab_patterns = {}
    try:
        with neo_driver.session() as session:
            print("--- 🔍 Extrayendo categorías de Neo4j ---")
            vocab_data = session.run(vocab_query).data()
            
            for row in vocab_data:
                root = row['root']
                words = [str(w) for w in row['words'] if w]
                vocab_patterns[root] = [re.compile(rf'\b{re.escape(w)}\b', re.IGNORECASE) for w in words]

            print(f"\n--- 📄 Analizando composición en % para {limit_pages} páginas ---")
            query_pages = "MATCH (p:Page) WHERE p.text IS NOT NULL RETURN p.text AS text LIMIT $limit"
            page_records = session.run(query_pages, limit=limit_pages).data()

        # CONTAR MENCIONES
        results = []
        for p in page_records:
            text = p['text']
            counts = {root: sum(len(pattern.findall(text)) for pattern in patterns) 
                      for root, patterns in vocab_patterns.items()}
            results.append(counts)

        df_counts = pd.DataFrame(results)
        df_counts = df_counts.loc[(df_counts != 0).any(axis=1)]

        # PASAR A PORCENTAJE (%)
        total_por_pagina = df_counts.sum(axis=1)
        df_porcentaje = (df_counts.div(total_por_pagina, axis=0) * 100).round(2)
        df_jittered = df_porcentaje.copy()
        df_jittered = df_jittered + np.random.uniform(-1.5, 1.5, size=df_jittered.shape)

        # GRAFICA
        sns.set_theme(style="white")
        columnas_activas = df_porcentaje.columns[(df_porcentaje > 1).any(axis=0)]
        df_final_plot = df_jittered[columnas_activas]

        g = sns.pairplot(
            df_final_plot, 
            kind='reg', 
            diag_kind='kde',
            corner=True,
            plot_kws={'line_kws': {'color': '#e53e3e', 'lw': 1.5}, 
                      'scatter_kws': {'alpha': 0.4, 'color': '#2f855a', 's': 20}},
            diag_kws={'color': '#2f855a', 'fill': True}
        )

        for ax in g.axes.flatten():
            if ax is not None:
                ax.set_xlim(-5, 105) 
                ax.set_ylim(-5, 105)
                ax.set_xlabel(ax.get_xlabel().upper() + " (%)", fontweight='bold', fontsize=10)
                ax.set_ylabel(ax.get_ylabel().upper() + " (%)", fontweight='bold', fontsize=10)

        g.fig.suptitle('Matriz de Dispersión por Página', 
                        y=1.03, fontsize=16, fontweight='bold')
        
        plt.show()
        
        return df_porcentaje

    except Exception as e:
        print(f"Error: {e}")
        return None

# EJECUTAR
df_tfg_porcentajes = generate_auto_intensity_matrix_porcentual(neo_driver)

La r al ser una regresión lineal nos calcula el mejor promedio, entonces hacer que la linea roja de r, empiece en 0,0 se podrñia pero sería forzando al 0,0 pq los datos no empiecen en 0, y lo mismo pasa con algunas lineas que están hacia abajo (negativo) que es q, por ejenplo, si drugs y carding tienen linea en negativo es q, cuanto mas hay de uno, menos hay de la otra, por eso es negativo.

In [ ]:
def generate_coexistence_matrix(neo_driver, limit_pages=3000, min_p=0.01):
    # --- PARTE 1: EXTRACCIÓN Y PROCESAMIENTO (Igual que el anterior) ---
    vocab_query = "MATCH (t:Term) OPTIONAL MATCH (s:Synonym)-[:IS_SYNONYM_OF]->(t) RETURN t.name AS root, collect(s.name) + t.name AS words"
    vocab_patterns = {}
    with neo_driver.session() as session:
        vocab_data = session.run(vocab_query).data()
        for row in vocab_data:
            root = row['root']
            words = [str(w) for w in row['words'] if w]
            vocab_patterns[root] = [re.compile(rf'\b{re.escape(w)}\b', re.IGNORECASE) for w in words]

        query_pages = "MATCH (p:Page) WHERE p.text IS NOT NULL RETURN p.text AS text LIMIT $limit"
        page_records = session.run(query_pages, limit=limit_pages).data()

    results = []
    for p in page_records:
        text = p['text']
        counts = {root: sum(len(pattern.findall(text)) for pattern in patterns) for root, patterns in vocab_patterns.items()}
        results.append(counts)

    df_counts = pd.DataFrame(results)
    df_counts = df_counts.loc[(df_counts != 0).any(axis=1)]
    total_por_pagina = df_counts.sum(axis=1)
    df_porcentaje = (df_counts.div(total_por_pagina, axis=0) * 100).round(2).fillna(0)

    # Filtrado de columnas por presencia global
    presencia_real = (df_counts > 0).mean()
    columnas_activas = presencia_real[presencia_real >= min_p].index.tolist()
    
    if len(columnas_activas) < 2:
        return print("No hay suficientes datos.")

    # --- PARTE 2: VISUALIZACIÓN PERSONALIZADA (FILTRANDO CEROS POR PAR) ---
    df_final = df_porcentaje[columnas_activas]
    num_vars = len(columnas_activas)
    
    fig, axes = plt.subplots(num_vars, num_vars, figsize=(num_vars*3, num_vars*3))
    plt.subplots_adjust(top=0.95, bottom=0.05, left=0.05, right=0.95, hspace=0.3, wspace=0.3)

    for i, row_col in enumerate(columnas_activas):
        for j, col_col in enumerate(columnas_activas):
            ax = axes[i, j]
            
            if i == j:  # Diagonal: Distribución de la categoría
                data_diag = df_final[row_col][df_final[row_col] > 0]
                if not data_diag.empty:
                    sns.kdeplot(data_diag, ax=ax, fill=True, color='#2f855a')
                ax.set_title(row_col.upper(), fontweight='bold')
            
            elif i > j: # Dispersión filtrada (COEXISTENCIA)
                # FILTRO DE COEXISTENCIA
                subset = df_final[(df_final[row_col] > 0) & (df_final[col_col] > 0)]
                
                if not subset.empty:
                    # Aplicamos jitter pequeño para que no se pisen
                    jitter_x = subset[col_col] + np.random.uniform(-1, 1, size=len(subset))
                    jitter_y = subset[row_col] + np.random.uniform(-1, 1, size=len(subset))
                    
                    sns.regplot(x=jitter_x, y=jitter_y, ax=ax, 
                                scatter_kws={'alpha':0.5, 's':15, 'color':'#2f855a'},
                                line_kws={'color':'#e53e3e', 'lw':1.5})
                
                ax.set_xlim(0, 105)
                ax.set_ylim(0, 105)
            else:
                ax.axis('off') # Ocultamos la parte superior (corner=True)

    plt.suptitle(f'Matriz de Coexistencia Semántica (Solo menciones conjuntas > {min_p*100:.1f}%)', 
                 fontsize=16, fontweight='bold')
    plt.show()

# Ejecución
generate_coexistence_matrix(neo_driver, min_p=0.2)

### Análisis de comunidades dentro de la Dark Web

En esta grafica se muestra un cluster que muestra las 5 comunidades más relevantes de la Dark Web y sus relaciones entre los nodos o similitudes con otras comunidades.  
Además se utilizan diversos filtros para poder tener un cluster legible.
- resolution=0.7, por lo tanto es menor a 1 entonces buscas las que sean más solidas las relaciones
- solo cogemos los nodos estructurales de la red
- quitamos el "ruido" es decir tenemos un filtro de 1% de la red para que si es muy poco entoncesss no lo coja y además que tengan las paginas mas de 1 conexión es decir de 2 conexiones para arriba para filtrar mas.

Los puntos del cluster dependen de la posición y significa lo siguiente:
- Si estan en el centro son puntos puente, es decir, se encargan de haceer de unión entre los diversos puntos, y eso es pq son paginas que contienen diversas tematicas y unen varias de ellas.
- Si estan mas a la izquierda o derecha, cuanto mas al extremo de arriba abajo izquierda o derecha significa que está mas aislado, es decir, que en su mayoria las paginas son independientes de esa tematica o tematicas, y no tienen mucha relación con otras páginas.
- Si un punto esta muy cerca o casi superpuesto a otro es por afinidad entre esos puntos, es decir, hay muchos links que comparten entre ellos, pero no están directamente relacionados entre ellos.
- Y los puntos que estan muy separados es decir, uno a la izquierda y otro muy a la derecha, es que son polos opuestos, es decir que no tienen muchos links o enlaces compartidos entre ellos.

In [ ]:
def percentage_identity_analysis(neo_driver, num_relaciones=5000):
    if not neo_driver:
        return print("❌ Error: Driver no disponible.")

    try:
        # NORMALIZAR DICCIONARIO
        colores_map = {}
        if 'colores_tfg' in globals():
            colores_map = {str(k).lower().strip(): v for k, v in colores_tfg.items()}
        else:
            print("⚠️ No veo la caja de 'colores_tfg' fuera.")

        with neo_driver.session() as session:
            print(f"📡 1. Analizando {num_relaciones} relaciones...")
            query = """
            MATCH (p1:Page)-[:LINKS_TO]->(p2:Page)
            WITH p1, p2 LIMIT $limit
            OPTIONAL MATCH (p1)-[:MENTIONS]->(s:Synonym)-[:IS_SYNONYM_OF]->(t:Term)
            RETURN p1.url as source, p2.url as target, collect(t.name) as terminos
            """
            result = session.run(query, limit=num_relaciones)
            G_raw = nx.Graph()
            dict_terminos = {}
            for record in result:
                u, v = record["source"], record["target"]
                G_raw.add_edge(u, v)
                if u not in dict_terminos: dict_terminos[u] = record["terminos"]

        # LOUVAIN Y FILTRADO
        nodos_relevantes = [n for n, d in G_raw.degree() if d > 1]
        G = G_raw.subgraph(nodos_relevantes).copy()
        particion_raw = community_louvain.best_partition(G, resolution=0.7, random_state=42)
        counts = pd.Series(particion_raw.values()).value_counts()
        umbral_minimo = max(5, int(G.number_of_nodes() * 0.01))
        comunidades_grandes = counts[counts >= umbral_minimo].index.tolist()
        particion = {n: p for n, p in particion_raw.items() if p in comunidades_grandes}
        G_final = G.subgraph(particion.keys())

        datos_clusters = []
        df_p = pd.DataFrame(list(particion.items()), columns=['url', 'comunidad'])
        
        for comm_id in comunidades_grandes:
            urls = df_p[df_p['comunidad'] == comm_id]['url']
            terminos_barrio = []
            for u in urls: terminos_barrio.extend(dict_terminos.get(u, []))
            
            if terminos_barrio:
                conteo = pd.Series(terminos_barrio).value_counts()
                total = conteo.sum()
                top_2 = conteo.head(2)
                tema_principal = str(top_2.index[0]).lower().strip()
                
                datos_clusters.append({
                    'comm_id': comm_id,
                    'tema_match': tema_principal, 
                    'perc_dom': (top_2.values[0] / total) * 100,
                    'etiqueta': " | ".join([f"{t.upper()} ({round((v/total)*100, 1)}%)" for t, v in top_2.items()]),
                    'paginas': len(urls),
                    'menciones': total,
                    'otros': f"{round(((total - top_2.sum())/total)*100, 1)}%"
                })

        # ASIGNAR LOS COLORES
        df_temp = pd.DataFrame(datos_clusters).sort_values(['tema_match', 'perc_dom'], ascending=[True, False])
        colores_finales_cluster = {}
        categorias_asignadas = set()
        paleta_extra = plt.get_cmap('Set3', 12) 

        for idx, row in df_temp.iterrows():
            tema = row['tema_match']
            if tema in colores_map and tema not in categorias_asignadas:
                colores_finales_cluster[row['comm_id']] = colores_map[tema]
                categorias_asignadas.add(tema)
            else:
                colores_finales_cluster[row['comm_id']] = mcolors.to_hex(paleta_extra(idx % 12))

        # GRAFICO
        plt.figure(figsize=(16, 12), facecolor='white')
        plt.suptitle('Clustering temático estructural (Algoritmo Louvain)', fontsize=18, fontweight='bold', y=0.95)
        pos = nx.spring_layout(G_final, k=0.5, seed=42)
        resumen = []
        legend_handles = []

        for i, comm_id in enumerate(comunidades_grandes, start=1):
            info = next(item for item in datos_clusters if item['comm_id'] == comm_id)
            color = colores_finales_cluster[comm_id]
            
            resumen.append({
                "ID Cluster": i, "Nº Páginas": info['paginas'],
                "Temática Dominante": info['tema_match'].upper(),
                "Composición Top 2 (%)": info['etiqueta'],
                "Otros Términos (%)": info['otros'], "Total Menciones": info['menciones']
            })
            legend_handles.append(mpatches.Patch(color=color, label=f"Cluster {i}: {info['etiqueta']}"))

        nx.draw_networkx_edges(G_final, pos, alpha=0.08, edge_color='gray')
        nx.draw_networkx_nodes(G_final, pos, 
                               node_size=[len(dict_terminos.get(n,[]))*70+50 for n in G_final.nodes()], 
                               node_color=[colores_finales_cluster[particion[n]] for n in G_final.nodes()], 
                               alpha=0.85)

        plt.legend(
            handles=legend_handles, 
            title="Identidad Dominante por Comunidad (Top 2 Términos)", 
            title_fontsize=13,            
            loc="upper center", 
            bbox_to_anchor=(0.5, -0.08),  
            ncol=2,                       
            frameon=True, 
            shadow=True,                  
            prop={'size': 11, 'weight': 'bold'}  
        )
        plt.axis('off')
        carpeta_destino = r"C:\Users\sandr\TFG\Analisis\Graficas"
        os.makedirs(carpeta_destino, exist_ok=True)
        print(f"📂 Los archivos se guardarán en: {os.path.abspath(carpeta_destino)}\n")
        
        # Guardamos la red en alta calidad (300 DPI) para evitar pixelados en Word/LaTeX
        path_imagen_tfg = os.path.join(carpeta_destino, "clustering_louvain_tematico.svg")
        plt.savefig(path_imagen_tfg, dpi=300, bbox_inches='tight')
        print(f"🖼️ Gráfica de red guardada con éxito en: {path_imagen_tfg}")
        plt.show()

        # TABLA
        df_final = pd.DataFrame(resumen).sort_values(by="ID Cluster")
        display(HTML("<style>.itables table { margin-left: 0 !important; width: 900px !important; text-align: left; } </style>"))
        show(df_final, paging=False, search=False, info=False, dom='t', autoWidth=False)
        
    except Exception as e:
        print(f"❌ Error: {e}")

# EJECUTAR
percentage_identity_analysis(neo_driver, num_relaciones=5000)

La siguiente grafica se encarga de medir la fuerza del vínculo o enlace entre dos terminos mediante el conteo de la frecuencia.  
Es decir, mira las páginas que tienen juntas A y B, las suma y lo indica, y luego lo mira con A y C por ejemplo y, dependiendo de la cantidad es una unión más fuerte o más debil.

In [ ]:
def analyze_term_cooccurrence(limit_pages=2000):
    global colores_tfg
    colores_map = {str(k).lower(): v for k, v in colores_tfg.items()}

    #CONSULTA
    query = """
    MATCH (p:Page)-[:MENTIONS]->(s:Synonym)-[:IS_SYNONYM_OF]->(t:Term)
    WITH p, collect(distinct t.name) as terminos
    WHERE size(terminos) > 1
    RETURN terminos
    LIMIT $limit
    """
    
    try:
        with neo_driver.session() as session:
            result = session.run(query, limit=limit_pages)
            
            pair_counts = {}
            term_totals = {} # Para saber cuántas veces aparece cada término solo
            
            for record in result:
                terminos = sorted(record["terminos"])
                for t in terminos:
                    term_totals[t] = term_totals.get(t, 0) + 1
                
                for pair in itertools.combinations(terminos, 2):
                    pair_counts[pair] = pair_counts.get(pair, 0) + 1
            
            if not pair_counts:
                return print("⚠️ No hay co-ocurrencias.")

            # CALCULO METRICAS
            data = []
            for (t_a, t_b), freq in pair_counts.items():
                # JACCARD PERO SIMPLIFICADO
                fuerza = freq / (term_totals[t_a] + term_totals[t_b] - freq)
                
                # Etiqueta de intensidad
                if fuerza > 0.6: intensidad = "MUY FUERTE"
                elif fuerza > 0.3: intensidad = "FUERTE"
                elif fuerza > 0.1: intensidad = "MODERADA"
                else: intensidad = "DÉBIL"
                
                data.append({
                    'Término A': t_a, 'Término B': t_b, 
                    'Frecuencia': freq, 'Fuerza': round(fuerza, 3),
                    'Intensidad': intensidad
                })

            df_cooc = pd.DataFrame(data).sort_values(by='Fuerza', ascending=False)

            # GRAFICA
            G = nx.Graph()
            df_top_graph = df_cooc.head(15)
            for _, row in df_top_graph.iterrows():
                G.add_edge(row['Término A'], row['Término B'], weight=row['Fuerza'])

            plt.figure(figsize=(12, 8))
            pos = nx.spring_layout(G, k=1) 
            node_colors = [colores_map.get(node.lower(), '#CBD5E0') for node in G.nodes()]
            weights = [G[u][v]['weight'] * 15 for u, v in G.edges()] # Grosor según fuerza
            
            nx.draw_networkx_edges(G, pos, width=weights, edge_color='#A0AEC0', alpha=0.4)
            nx.draw_networkx_nodes(G, pos, node_size=4000, node_color=node_colors, edgecolors='white', linewidths=2)
            nx.draw_networkx_labels(G, pos, font_size=9, font_weight='bold')

            plt.title("Red de Asociaciones Temáticas (Grosor = Fuerza de Vínculo)", pad=20)
            plt.axis('off')
            plt.show()

            # TABLA
            print("\n📊 ANÁLISIS DE ASOCIACIONES (Ordenado por Fuerza):")
            show(df_cooc.head(20), paging=False, search=False, info=False, dom='t')
            
            return df_cooc

    except Exception as e:
        print(f"❌ Error: {e}")

df_semantica = analyze_term_cooccurrence(2000)

Lo siguiente que se quiere realizar es un estudio de los datos utilizando k-means, pero para poder realizarlo se necesita saber el número de clusteres que es más recomendado tener para la muestra de datos.  
Para ello se utiliza un método de validación de la k, que en este caso se realiza mediante 2 tipos distintos de graficas:
- Método del codo (Inertia): Identifica el número óptimo de clústeres observando el punto donde la reducción de la varianza interna deja de ser significativa, formando una curvatura característica en la gráfica.
- Análisis de la Silueta (Silhouette): Mide qué tan similar es un objeto a su propio clúster en comparación con otros, donde el valor más alto indica que la estructura de agrupamiento es sólida y los clústeres están bien separados.  
Y como se puede ver, la K elegida es 6.

In [ ]:
def validation_k(neo_driver, limite=5000):
    print("⚙️ Extrayendo datos y calculando composición porcentual...")
    
    # CONSULTA
    with neo_driver.session() as session:
        res_terms = session.run("MATCH (t:Term) RETURN t.name as nombre")
        mis_terminos = [r['nombre'].lower() for r in res_terms]
        
        res_pages = session.run("MATCH (p:Page) WHERE p.text IS NOT NULL RETURN p.url as url, p.text as text LIMIT $limit", limit=limite)
        data = res_pages.data()
        df = pd.DataFrame(data)

    if df.empty:
        print("❌ No hay datos."); return None, None, None

    # FRECUENCIA RELATIVA
    cv = CountVectorizer(vocabulary=mis_terminos, token_pattern=r'\b\w+\b')
    X_counts = cv.fit_transform(df['text']).toarray()
    sumas = X_counts.sum(axis=1)
    sumas[sumas == 0] = 1 
    X_relativo = (X_counts / sumas[:, None]) * 100

    print("📉 Calculando métricas K-Means (Codo y Silueta)...")
    inercia = []
    silueta = []
    K_range = range(2, 11) 

    for k in K_range:
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = km.fit_predict(X_relativo)
        inercia.append(km.inertia_)
        silueta.append(silhouette_score(X_relativo, labels))
    
    # GRAFICAS
    plt.figure(figsize=(16, 6)) 
    
    k_elegida = 6

    # PRIMERA GRAFICA: CODO
    plt.subplot(1, 2, 1)
    plt.plot(K_range, inercia, marker='o', color='royalblue', linewidth=2, label='Inercia')
    plt.axvline(x=k_elegida, color='red', linestyle='--', linewidth=2, label=f'K elegido ({k_elegida})')
    
    plt.title('Método del Codo (Inercia)', fontweight='bold', fontsize=13)
    plt.xlabel('Número de Clusters (K)')
    plt.ylabel('Inercia')
    plt.legend() 
    plt.grid(True, linestyle='--', alpha=0.4)
    
    # SEGUNDA GRAFICA: SILUETA
    plt.subplot(1, 2, 2)
    plt.plot(K_range, silueta, marker='s', color='forestgreen', linewidth=2, label='Silhouette Score')
    plt.axvline(x=k_elegida, color='red', linestyle='--', linewidth=2, label=f'K elegido ({k_elegida})')
    
    plt.title('Análisis de Silueta (Calidad)', fontweight='bold', fontsize=13)
    plt.xlabel('Número de Clusters (K)')
    plt.ylabel('Score (Más alto es mejor)')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.4)
    
    # GUARDAR
    ruta_dir = 'Graficas'
    os.makedirs(ruta_dir, exist_ok=True)
    nombre_archivo = 'validacion_kmeans_codo_silueta.svg'
    path_completo = os.path.join(ruta_dir, nombre_archivo)
    
    plt.tight_layout()
    plt.savefig(path_completo, dpi=300, bbox_inches='tight')
    print(f"✅ Gráficas con marca K={k_elegida} guardadas en: {path_completo}")
    plt.show()

    return df, X_relativo, cv

# EJECUCIÓN
df_prep, X_rel, cv_obj = validation_k(neo_driver, limite=5000)

Una vez se tiene la K elegida mediante los métodos anteriores, se realiza el análisis con K-Means que es un algoritmo de aprendizaje no supervisado que particiona la muestra de datos en clústeres, agrupando las páginas web según la similitud en su composición temática.  
Posterior a esto, se realiza un análisis de nube de palabras para cada uno de los clusters, para poder ver que incluye cada uno de ellos y una composición de cada cluster, inidicando el porcentaje de cada término en cada cluster.   
Finalmente se puede ver información de la anomalía detectada en el análisis de K-means para saber más información sobre ello.

Se eligió K-Means porque también se probó DBSCAN y Jerárquico, y, al ser iguales, la mejor opción era K-Means.

In [ ]:
def clustering_with_k(df_prep, X_relativo, cv, k_elegido=6):
    print(f"🤖 Iniciando análisis avanzado de Clústeres (K={k_elegido})...")
    
    # CLUSTERING
    kmeans = KMeans(n_clusters=k_elegido, random_state=42, n_init=10)
    df_prep['cluster'] = kmeans.fit_predict(X_relativo) + 1
    feature_names = cv.get_feature_names_out()

    # PCA Y ANOMALÍA
    pca = PCA(n_components=2)
    coords = pca.fit_transform(X_relativo)
    centro_masa = coords.mean(axis=0)
    distancias = np.linalg.norm(coords - centro_masa, axis=1)
    idx_anomalia = np.argmax(distancias)

    ruta_dir = 'Graficas'
    os.makedirs(ruta_dir, exist_ok=True)

    # GRAFICA
    plt.figure(figsize=(12, 7), facecolor='none')
    ax = plt.gca()
    ax.set_facecolor('none')
    colores_palette = sns.color_palette('magma', n_colors=k_elegido)
    
    for i, cluster_id in enumerate(sorted(df_prep['cluster'].unique())):
        puntos_cluster = coords[df_prep['cluster'] == cluster_id]
        if len(puntos_cluster) >= 3:
            hull = ConvexHull(puntos_cluster)
            poly = Polygon(puntos_cluster[hull.vertices], fill=True, 
                           facecolor=colores_palette[i], edgecolor=colores_palette[i], 
                           alpha=0.15, linewidth=1.5, linestyle='--', label='_nolegend_')
            ax.add_patch(poly)

    sns.scatterplot(x=coords[:,0], y=coords[:,1], hue=df_prep['cluster'], 
                    palette='magma', s=100, alpha=0.8, edgecolor='white')
    
    plt.scatter(coords[idx_anomalia, 0], coords[idx_anomalia, 1], color='red', 
                s=300, marker='X', label='Página Crítica (Anomalía)', edgecolor='black', zorder=10)
    
    plt.title(f'Segmentación Temática PCA (K={k_elegido})', fontsize=15, fontweight='bold', pad=20)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', frameon=False)
    plt.grid(True, alpha=0.2, linestyle='--')
    sns.despine()
    plt.savefig(os.path.join(ruta_dir, 'mapa_pca_envolventes.svg'), dpi=300, bbox_inches='tight')
    plt.show()

    # NUBE PALABRAS
    print("☁️ Generando Nubes de Palabras por Clúster...")
    plt.figure(figsize=(20, 6))
    for i in range(1, k_elegido + 1):
        mask = df_prep['cluster'] == i
        if mask.any():
            promedios_pct = X_relativo[mask].mean(axis=0)
            if hasattr(promedios_pct, "to_numpy"): promedios_pct = promedios_pct.to_numpy()
            frecuencias = dict(zip(feature_names, promedios_pct))
            frecuencias = {k: v for k, v in frecuencias.items() if v > 0}
            if frecuencias:
                wc = WordCloud(width=400, height=400, background_color='white', colormap='Dark2').generate_from_frequencies(frecuencias)
                plt.subplot(1, k_elegido, i)
                plt.imshow(wc, interpolation='bilinear')
                plt.title(f"Clúster {i}", fontweight='bold', fontsize=14)
                plt.axis('off')
    plt.tight_layout()
    plt.savefig(os.path.join(ruta_dir, 'wordclouds_clusters.svg'), dpi=300, bbox_inches='tight')
    plt.show()

    # TABLA
    print("\n📊 COMPOSICIÓN DETALLADA POR CLÚSTER:")
    
    df_temp = pd.DataFrame(X_relativo, columns=feature_names)
    df_temp['Cluster_ID'] = df_prep['cluster'].values
    tabla_composicion = df_temp.groupby('Cluster_ID').mean()
    suma_categorias = tabla_composicion.sum(axis=1)
    tabla_composicion['Otros'] = (100 - suma_categorias).clip(lower=0)
    tabla_composicion = tabla_composicion.round(2)

    df_visualizacion = tabla_composicion.astype(str) + "%"
    df_visualizacion.insert(0, 'Págs', df_prep.groupby('cluster').size())
    display(HTML("""
        <style>
            .itables table { 
                margin-left: 0 !important; 
                margin-right: auto !important; 
                width: auto !important;
                min-width: 800px !important;
                font-size: 0.82em !important;
                border-collapse: collapse !important;
            }
            .itables th { 
                background-color: #f7fafc !important; 
                color: #4a5568 !important; 
                padding: 10px 15px !important; 
                border-bottom: 2px solid #edf2f7 !important; 
                text-align: left !important;
            }
            .itables td { 
                padding: 8px 15px !important; 
                color: #2d3748 !important; 
                border-bottom: 1px solid #edf2f7 !important; 
            }
        </style>
    """))
    show(df_visualizacion, 
         paging=False, 
         search=False, 
         info=False, 
         dom='t', 
         scrollX=True,
         autoWidth=False,
         columnDefs=[
             {"width": "70px", "targets": 0},  # Columna Págs
             {"width": "100px", "targets": "_all"}, 
             {"className": "dt-left", "targets": "_all"}
         ])

    # GUARDAR
    tabla_csv = tabla_composicion.copy()
    if 'Págs' not in tabla_csv.columns:
        tabla_csv.insert(0, 'Págs', df_prep.groupby('cluster').size())
        
    csv_str = tabla_csv.to_csv(index=True)
    b64_csv = base64.b64encode(csv_str.encode()).decode()
    payload_csv = f"data:text/csv;base64,{b64_csv}"
    
    display(HTML(f"""
        <div style="text-align: left; margin-top: 15px; margin-bottom: 25px;">
            <span style="font-size: 10px; color: #718096; display: block; margin-bottom: 8px;">📁 Guardado en: {os.path.join(ruta_dir, 'composicion_clusters.csv')}</span>
            <a href="{payload_csv}" download="composicion_detallada_clusters.csv" style="
                display: inline-block; background-color: #38a169; color: white;
                padding: 6px 14px; text-decoration: none; border-radius: 6px;
                font-weight: 600; font-family: sans-serif; font-size: 12px;
            ">📥 Descargar Composición CSV</a>
        </div>
    """))

    # ANOMALÍA
    url_anomala = df_prep.iloc[idx_anomalia]['url']
    raw_text = df_prep.iloc[idx_anomalia]['text']
    soup = BeautifulSoup(raw_text, "html.parser")
    texto_limpio = "\n".join([l.strip() for l in soup.get_text(separator='\n').split('\n') if l.strip()][:50]) # Limitado para no saturar

    vec_anom = X_relativo.iloc[idx_anomalia] if hasattr(X_relativo, "iloc") else X_relativo[idx_anomalia]
    menciones = sorted(dict(zip(feature_names, vec_anom)).items(), key=lambda x: x[1], reverse=True)[:5]
    resumen_html = "".join([f"<li><b>{t.lower()}:</b> {v:.2f}%</li>" for t, v in menciones if v > 0])

    html_content = f"<html><body style='font-family:sans-serif;'><h1>Evidencia Onion</h1><p><b>URL:</b> {url_anomala}</p><ul>{resumen_html}</ul><hr><pre>{texto_limpio}</pre></body></html>"
    b64_html = base64.b64encode(html_content.encode()).decode()
    payload_html = f"data:text/html;base64,{b64_html}"

    display(HTML(f"""
        <div style="background: #ebf8ff; border: 1px solid #3182ce; padding: 15px; border-radius: 8px; max-width: 600px; margin-top: 10px;">
            <h4 style="color: #2b6cb0; margin: 0 0 10px 0;">🔎 Hallazgo Crítico (Anomalía)</h4>
            <p style="font-size: 12px; color: #2d3748;">Detectada página con perfil atípico en la distribución temática.</p>
            <a href="{payload_html}" download="evidencia_anomalia.html" style="
                display: inline-block; background-color: #3182ce; color: white;
                padding: 6px 14px; text-decoration: none; border-radius: 6px;
                font-weight: 600; font-family: sans-serif; font-size: 12px;
            ">📥 Descargar Evidencia HTML</a>
        </div>
    """))

    return df_prep

# EJECUTAR
df_final_resultados = clustering_with_k(df_prep, X_rel, cv_obj, k_elegido=6)

Análisis de redes complejas utilizando el método de Louvain para la detección de comunidades, lo hace por estructura de relaciones es decir, páginas conectadas entre sí. \
El resultado arrojó una modularidad sobresaliente de 0.939, lo que indica una estructura de red altamente compartimentada.
Se identificaron 35 comunidades diferenciadas, lo que sugiere que la actividad en la muestra de la Dark Web analizada se organiza en sub-nichos especializados y aislados, permitiendo identificar patrones de comportamiento grupal que van más allá de la simple clasificación temática.

In [ ]:
def analysis_complex_networks_louvain_FULL(df_datos, X_relativo, cv):
    """
    Detecta comunidades mediante Louvain y caracteriza su ADN temático.
    """
    print("🕸️ 1. Iniciando Detección de Comunidades (Louvain)...")
    
    # --- PASO 1: CONSTRUCCIÓN DEL GRAFO ---
    G = nx.Graph()
    col_cluster = 'cluster' if 'cluster' in df_datos.columns else df_datos.columns[-1]
    df_trabajo = df_datos.reset_index(drop=True)
    
    for _, grupo in df_trabajo.groupby(col_cluster):
        urls = grupo['url'].tolist()
        for i in range(len(urls) - 1):
            G.add_edge(urls[i], urls[i+1])

    # --- PASO 2: ALGORITMO LOUVAIN ---
    particion = community_louvain.best_partition(G)
    mod = community_louvain.modularity(particion, G)
    
    # --- PASO 3: MAPA GLOBAL ---
    plt.figure(figsize=(15, 10))
    pos = nx.spring_layout(G, k=0.5, iterations=50, seed=42)
    nx.draw_networkx_nodes(G, pos, node_size=80, node_color=list(particion.values()), 
                           cmap='tab20', edgecolors='white', linewidths=0.5, alpha=0.9)
    nx.draw_networkx_edges(G, pos, alpha=0.05, edge_color='gray')
    plt.title(f"Mapa Estructural Global (Modularidad: {mod:.3f})", fontsize=15, fontweight='bold')
    plt.axis('off')
    plt.show()

    # --- PASO 4: CARACTERIZACIÓN ADN ---
    df_trabajo['comunidad_id'] = df_trabajo['url'].map(particion)
    df_trabajo = df_trabajo.dropna(subset=['comunidad_id'])
    terminos = cv.get_feature_names_out()
    reporte_list = []

    for comu_id in sorted(df_trabajo['comunidad_id'].unique()):
        indices_barrio = df_trabajo[df_trabajo['comunidad_id'] == comu_id].index
        matriz_barrio = X_relativo[indices_barrio]
        centroide = matriz_barrio.mean(axis=0)
        if hasattr(centroide, "A1"): centroide = centroide.A1 
        
        indices_con_valor = np.where(centroide > 1e-7)[0]
        if len(indices_con_valor) > 0:
            sorted_idx = indices_con_valor[np.argsort(centroide[indices_con_valor])][::-1]
            top_idx = sorted_idx[:3]
            # Formato en minúsculas y con (%) para el TFG
            terminos_str = " | ".join([f"{terminos[i]} ({centroide[i]:.1f}%)" for i in top_idx])
        else:
            terminos_str = "perfil genérico"

        reporte_list.append({
            'ID Comunidad': int(comu_id),
            'Nº Páginas': len(indices_barrio),
            'ADN Temático Principal': terminos_str
        })
            
    df_reporte = pd.DataFrame(reporte_list).sort_values(by='Nº Páginas', ascending=False)
    return G, particion, df_reporte

# --- 2. FUNCIÓN DE VISTA LIMPIA CON LEYENDA ---
def plot_vista_limpia_con_leyenda(G, particion, df_reporte, top_n=6):
    print(f"\n🧹 Generando vista de las {top_n} comunidades principales con leyenda...")

    top_df = df_reporte.head(top_n)
    top_comus = top_df['ID Comunidad'].tolist()
    nodes_to_keep = [n for n, com in particion.items() if com in top_comus]
    G_sub = G.subgraph(nodes_to_keep)

    plt.figure(figsize=(16, 10))
    pos = nx.spring_layout(G_sub, k=1.2, iterations=100, seed=42)
    cmap = plt.get_cmap('tab10')

    comu_to_color_idx = {com_id: i for i, com_id in enumerate(top_comus)}
    node_colors = [cmap(comu_to_color_idx[particion[n]]) for n in G_sub.nodes()]
    
    nx.draw_networkx_nodes(G_sub, pos, node_size=180, 
                           node_color=node_colors, 
                           edgecolors='white', linewidths=1.2)
    
    nx.draw_networkx_edges(G_sub, pos, alpha=0.15, edge_color='gray')

    # --- CONSTRUCCIÓN DE LA LEYENDA ---
    leyenda_patches = []
    for i, (_, row) in enumerate(top_df.iterrows()):
        comu_id = row['ID Comunidad']
        # Formateamos el ADN para que sea legible
        adn_limpio = row['ADN Temático Principal'].replace(' | ', '\n   ')
        
        patch = mpatches.Patch(color=cmap(i), 
                               label=f"Comunidad {comu_id}:\n   {adn_limpio}\n   ({int(row['Nº Páginas'])} págs)")
        leyenda_patches.append(patch)

    plt.legend(handles=leyenda_patches, 
           title="Perfil Temático de Barrios (Top)", 
           title_fontsize='13',
           loc='center left', 
           bbox_to_anchor=(1, 0.5), 
           fontsize=9,
           frameon=True,
           shadow=True)

    plt.title(f"Análisis Estructural de Sub-comunidades (Top {top_n})", fontsize=16, fontweight='bold', pad=25)
    plt.axis('off')
    plt.savefig("vista_comunidades_tfg.svg", dpi=300, bbox_inches='tight')
    plt.show()

# --- EJECUCIÓN TOTAL ---

# 1. Ejecutar detección y reporte
G_obj, particion_final, tabla_reporte = analysis_complex_networks_louvain_FULL(df_final_resultados, X_rel, cv_obj)

# 2. Generar gráfico con leyenda integrada
plot_vista_limpia_con_leyenda(G_obj, particion_final, tabla_reporte, top_n=6)

# --- 3. MOSTRAR TABLA DE DATOS (ESTILO PREMIUM REINA) ---
print("\n📊 CARACTERIZACIÓN DETALLADA DE BARRIOS:")

# Creamos una copia limpia para la visualización
df_tabla_redes = tabla_reporte.head(10).copy()

display(HTML("""
    <style>
        .itables table { 
            margin-left: 0 !important; 
            margin-right: auto !important; 
            width: 700px !important; 
            max-width: 800px !important;
            font-size: 0.85em !important;
            border-collapse: collapse !important;
            table-layout: fixed !important; 
        }
        .itables th, .itables td { 
            padding: 10px 15px !important; 
            white-space: nowrap !important;
            overflow: hidden;
            text-overflow: ellipsis;
        }
        .itables th { background-color: #f7fafc !important; border-bottom: 2px solid #edf2f7 !important; }
        .itables td { border-bottom: 1px solid #edf2f7 !important; }
    </style>
    """))
show(df_tabla_redes, 
     paging=False, 
     search=False, 
     info=False, 
     dom='t',
     autoWidth=False,
     columnDefs=[
         {"width": "120px", "targets": 0},
         {"width": "120px", "targets": 1},
         {"width": "460px", "targets": 2}, 
         {"className": "dt-left", "targets": "_all"}
     ])

### Análisis con Spearman

Aquí se realiza un diagnostico de la muestra con Spearman que separa temas que simplemente son ruido de otros que son nucleo (es decir, que están conectados).  
Utilizamos 3 caracteristicas con esos datos:
- **Densidad:** Ignora el término si no tiene minimo el 40% de la página.
- **Popularidad:** Mira en cuantas páginas sale el término como el dominante de la página (mayor porcentaje de la página).
- **Centalidad:** Calcula la conectividad (si tiene 100% es que está vinclulado a todos los demás términos).

La gráfica tiene el siguiente significado:
- **Barra:** Conteo bruto de páginas donde el término dado es dominante.
- **Línea:** Indica si el término es transversal, es decir, si conecta con otros términos o no.

In [ ]:
def centrality_analysis_pro(neo_driver, umbral_pct=0.40, min_cooc_paginas=5):
    """
    Analiza la relevancia estructural de los términos usando porcentajes de 
    conectividad relativa en lugar de valores absolutos.
    """
    print(f"🔍 Iniciando Diagnóstico Estructural (Umbral de presencia > {umbral_pct*100}%)...")
    
    # CONSULTA
    query_raw = """
    MATCH (p:Page)-[:MENTIONS]->(s:Synonym)-[:IS_SYNONYM_OF]->(t:Term)
    RETURN p.url as url, t.name as termino, count(s) as n_veces
    """
    
    try:
        with neo_driver.session() as session:
            result = session.run(query_raw)
            df_raw = pd.DataFrame([dict(r) for r in result])
            
        if df_raw.empty:
            print("❌ No se encontraron datos en la base de datos."); return None

        # FILTRADO
        df_raw = df_raw.groupby(['url', 'termino'], as_index=False)['n_veces'].sum()
        
        # FILTRO 1: DOMINANCIA
        total_por_pagina = df_raw.groupby('url')['n_veces'].transform('sum')
        df_raw['pct_en_pagina'] = (df_raw['n_veces'] / total_por_pagina) * 100
        df_filtrado = df_raw[df_raw['pct_en_pagina'] >= (umbral_pct * 100)].copy()
        
        # POPULARIDAD
        popularidad = df_filtrado.groupby('termino')['url'].nunique().reset_index(name='Popularidad')
        
        # CENTALIDAD
        df_links = pd.merge(df_filtrado, df_filtrado, on='url', suffixes=('_A', '_B'))
        df_links = df_links[df_links['termino_A'] != df_links['termino_B']]
        cooc = df_links.groupby(['termino_A', 'termino_B'])['url'].nunique().reset_index(name='count')
        cooc_fuerte = cooc[cooc['count'] >= min_cooc_paginas]
        centralidad_abs = cooc_fuerte.groupby('termino_A')['termino_B'].nunique().reset_index(name='n_conexiones')
        df_final = pd.merge(popularidad, centralidad_abs, left_on='termino', right_on='termino_A', how='left').fillna(0)
        
        # CALCULO DE CONECTIVIDAD
        n_total_terminos = df_final['termino'].nunique()
        max_posible = n_total_terminos - 1
        
        df_final['Conectividad_Relativa'] = (df_final['n_conexiones'] / max_posible) * 100
        df_final = df_final.sort_values(by='Popularidad', ascending=False).drop(columns=['termino_A', 'n_conexiones'])

        # GRAFICA
        plt.figure(figsize=(14, 8))
        ax1 = plt.gca()
        lista_colores = [colores_tfg.get(t.lower(), '#333') for t in df_final['termino']]

        # BARRAS
        sns.barplot(data=df_final, x="termino", y="Popularidad", ax=ax1, palette=lista_colores, alpha=0.8, hue="termino", legend=False)
        ax1.set_ylabel('Popularidad (Nº Páginas)', fontsize=12, fontweight='bold', color='#4A5568')
        ax1.set_xlabel('')
        plt.xticks(rotation=45, ha='right', fontweight='bold')

        # LÍNEAS
        ax2 = ax1.twinx()
        sns.lineplot(data=df_final, x="termino", y="Conectividad_Relativa", ax=ax2, 
                     marker='o', markersize=12, color='#2D3748', linewidth=4, label='Índice de Conectividad (%)')
        
        ax2.set_ylabel('Conectividad Relativa (%)', fontsize=12, fontweight='bold', color='#2D3748')
        ax2.set_ylim(0, 110) 
        ax2.grid(False)

        for i, val in enumerate(df_final['Conectividad_Relativa']):
            ax2.text(i, val + 5, f"{val:.0f}%", color='#2D3748', ha='center', fontweight='bold', fontsize=11)

        plt.title(f"Mapa de Relevancia: Volumen vs. Interconectividad (Umbral {umbral_pct*100}%)", fontsize=16, fontweight='bold', pad=25)
        
        # GUARDADO
        os.makedirs('Graficas', exist_ok=True)
        plt.savefig('Graficas/diagnostico_relativo_tfg.svg', dpi=300, bbox_inches='tight')
        plt.show()

        # TABLA
        print("\n📋 REPORTE DE IMPORTANCIA ESTRUCTURAL:")
        df_tabla = df_final.copy()
        df_tabla['Conectividad_Relativa'] = df_tabla['Conectividad_Relativa'].map('{:.1f}%'.format)
        
        display(HTML("<style>.itables table { margin-left: 0 !important; width: 600px !important; }</style>"))
        show(df_tabla, paging=False, search=False, info=False, dom='t')

        return df_final

    except Exception as e:
        print(f"❌ Error en el análisis: {e}")
        return None
        
# EJECUTAR
df_diagnostico = centrality_analysis_pro(neo_driver, umbral_pct=0.40)

In [ ]:
def centrality_analysis_pro(neo_driver, umbral_pct=0.40, min_cooc_paginas=5):
    """
    Analiza la relevancia estructural de los términos usando porcentajes de 
    conectividad relativa en lugar de valores absolutos.
    """
    print(f"🔍 Iniciando Diagnóstico Estructural (Umbral de presencia > {umbral_pct*100}%)...")
    
    # CONSULTA
    query_raw = """
    MATCH (p:Page)-[:MENTIONS]->(s:Synonym)-[:IS_SYNONYM_OF]->(t:Term)
    RETURN p.url as url, t.name as termino, count(s) as n_veces
    """
    
    try:
        with neo_driver.session() as session:
            result = session.run(query_raw)
            df_raw = pd.DataFrame([dict(r) for r in result])
            
        if df_raw.empty:
            print("❌ No se encontraron datos en la base de datos."); return None

        # FILTRADO
        df_raw = df_raw.groupby(['url', 'termino'], as_index=False)['n_veces'].sum()
        
        # FILTRO 1: DOMINANCIA
        total_por_pagina = df_raw.groupby('url')['n_veces'].transform('sum')
        df_raw['pct_en_pagina'] = (df_raw['n_veces'] / total_por_pagina) * 100
        df_filtrado = df_raw[df_raw['pct_en_pagina'] >= (umbral_pct * 100)].copy()
        
        # POPULARIDAD
        popularidad = df_filtrado.groupby('termino')['url'].nunique().reset_index(name='Popularidad')
        
        # CENTRALIDAD
        df_links = pd.merge(df_filtrado, df_filtrado, on='url', suffixes=('_A', '_B'))
        df_links = df_links[df_links['termino_A'] != df_links['termino_B']]
        cooc = df_links.groupby(['termino_A', 'termino_B'])['url'].nunique().reset_index(name='count')
        cooc_fuerte = cooc[cooc['count'] >= min_cooc_paginas]
        centralidad_abs = cooc_fuerte.groupby('termino_A')['termino_B'].nunique().reset_index(name='n_conexiones')
        df_final = pd.merge(popularidad, centralidad_abs, left_on='termino', right_on='termino_A', how='left').fillna(0)
        
        # CALCULO DE CONECTIVIDAD
        n_total_terminos = df_final['termino'].nunique()
        max_posible = n_total_terminos - 1
        
        df_final['Conectividad_Relativa'] = (df_final['n_conexiones'] / max_posible) * 100
        df_final = df_final.sort_values(by='Popularidad', ascending=False).drop(columns=['termino_A', 'n_conexiones'])

        # =========================================================================
        # CONFIGURACIÓN DE GRÁFICA MIXTA EN TAMAÑO GIGANTE (POWERPOINT/PDF)
        # =========================================================================
        plt.figure(figsize=(16, 10)) # Lienzo más amplio para que no se apelmace el texto
        ax1 = plt.gca()
        lista_colores = [colores_tfg.get(t.lower(), '#333') for t in df_final['termino']]

        # 1. EJECUCIÓN Y AJUSTE DE BARRAS (EJE IZQUIERDO)
        sns.barplot(data=df_final, x="termino", y="Popularidad", ax=ax1, palette=lista_colores, alpha=0.75, hue="termino", legend=False)
        
        ax1.set_ylabel('Popularidad (Nº Páginas)', fontsize=14, fontweight='bold', color='#4A5568', labelpad=15)
        ax1.tick_params(axis='y', labelsize=13)
        ax1.set_xlabel('')
        
        # Ajuste nombres de los términos (Eje X)
        ax1.set_xticklabels(ax1.get_xticklabels(), rotation=45, ha='right', fontweight='bold', fontsize=13)

        # 2. EJECUCIÓN Y AJUSTE DE LÍNEA ESTRUCTURAL (EJE DERECHO DOUBLE-Y)
        ax2 = ax1.twinx()
        sns.lineplot(data=df_final, x="termino", y="Conectividad_Relativa", ax=ax2, 
                     marker='o', markersize=14, color='#2D3748', linewidth=4.5, label='Índice de Conectividad (%)')
        
        ax2.set_ylabel('Conectividad Relativa (%)', fontsize=14, fontweight='bold', color='#2D3748', labelpad=15)
        ax2.tick_params(axis='y', labelsize=13)
        ax2.set_ylim(0, 115) # Un poco más de holgura superior para que las etiquetas de texto no se corten
        ax2.grid(False) # Evitamos que duplique líneas de rejilla

        # ETIQUETAS PORCENTUALES FLOTANTES (Letras grandes)
        for i, val in enumerate(df_final['Conectividad_Relativa']):
            ax2.text(i, val + 4, f"{val:.0f}%", color='#2D3748', ha='center', fontweight='bold', fontsize=12)

        # CONFIGURACIÓN DE TÍTULOS Y LEYENDA FLOTANTE
        plt.title(f"Mapa de Relevancia: Volumen vs. Interconectividad (Umbral {umbral_pct*100}%)", 
                  fontsize=18, fontweight='bold', pad=30)
        
        ax2.legend(loc='upper right', fontsize=12, frameon=True, shadow=True)
        
        ax1.grid(axis='y', linestyle='--', alpha=0.3)
        sns.despine(ax=ax1, right=False) # Mantiene la línea del eje derecho visible
        
        plt.tight_layout()

        # GUARDADO VECTORIAL PREMIUM
        os.makedirs('Graficas', exist_ok=True)
        plt.savefig('Graficas/diagnostico_relativo_tfg.svg', bbox_inches='tight')
        plt.show()

        # TABLA
        print("\n📋 REPORTE DE IMPORTANCIA ESTRUCTURAL:")
        df_tabla = df_final.copy()
        df_tabla['Conectividad_Relativa'] = df_tabla['Conectividad_Relativa'].map('{:.1f}%'.format)
        
        display(HTML("<style>.itables table { margin-left: 0 !important; width: 600px !important; }</style>"))
        show(df_tabla, paging=False, search=False, info=False, dom='t')

        return df_final

    except Exception as e:
        print(f"❌ Error en el análisis: {e}")
        return None
        
# EJECUTAR
df_diagnostico = centrality_analysis_pro(neo_driver, umbral_pct=0.40)

### Análisis de dependencias bayesianas

Análisis de dependencias basado en redes bayesianas, es decir, la probabilidad de que a partir de un término A, aparezca un término B. \
No es bidireccional, puede que de A a B sea una probabilidad alta pero al reves sea baja. \
Aquí solo vemos las que tiene una probabilidad de 0.70 o mayor, ya que sino hay muchas relaciones y no podremos ver bien toda la información. 

In [ ]:
def bayesian_network_of_dependencies(neo_driver):
    print("🧮 Calculando probabilidades condicionales P(B|A)...")

    # CONSULTA
    query = """
    MATCH (p:Page)-[:MENTIONS]->(t:Term)
    RETURN p.url as doc_id, t.name as term_name
    UNION
    MATCH (p:Page)-[:MENTIONS]-(s:Synonym)-[:IS_SYNONYM_OF]->(t:Term)
    RETURN p.url as doc_id, t.name as term_name
    """
    
    try:
        with neo_driver.session() as session:
            result = session.run(query)
            datos = [dict(r) for r in result]
            
            if not datos: return None
            df_raw = pd.DataFrame(datos)
            if 'doc_id' not in df_raw.columns: return None

        presencia = pd.crosstab(df_raw['doc_id'], df_raw['term_name'])
        presencia = (presencia > 0).astype(int)
        
        nombres_terminos = presencia.columns
        matrix_data = []

        for term_a in nombres_terminos:
            fila = []
            count_a = presencia[term_a].sum()
            for term_b in nombres_terminos:
                if term_a == term_b:
                    fila.append(0.0)
                else:
                    ambos = ((presencia[term_a] == 1) & (presencia[term_b] == 1)).sum()
                    prob = ambos / count_a if count_a > 0 else 0
                    fila.append(prob)
            matrix_data.append(fila)
        
        return pd.DataFrame(matrix_data, index=nombres_terminos, columns=nombres_terminos)

    except Exception as e:
        print(f"❌ Error: {e}")
        return None

def plot_bayesian_with_table_colored_save(dep_matrix, threshold=0.70):
    if dep_matrix is None or dep_matrix.empty:
        return None
        
    G = nx.DiGraph() 
    relaciones_lista = []

    for term_a in dep_matrix.index:
        for term_b in dep_matrix.columns:
            prob = dep_matrix.loc[term_a, term_b]
            if prob >= threshold:
                G.add_edge(term_a, term_b, weight=prob)
                relaciones_lista.append({
                    'Causa (A)': term_a.upper(),
                    'Efecto (B)': term_b.upper(),
                    'Probabilidad': float(prob)
                })

    if not relaciones_lista:
        print(f"ℹ️ Ninguna relación supera el {threshold*100}%.")
        return pd.DataFrame()

    # GRÁFICA
    plt.figure(figsize=(14, 11))
    pos = nx.circular_layout(G)
    
    node_colors = [colores_tfg.get(node.lower(), '#999') for node in G.nodes()]
    edge_colors = [colores_tfg.get(u.lower(), 'black') for u, v in G.edges()]
    
    nx.draw_networkx_nodes(G, pos, node_size=5500, node_color=node_colors, 
                           edgecolors='black', linewidths=2, alpha=1.0)
    
    nx.draw_networkx_labels(G, pos, font_size=11, font_weight='bold',
                            bbox=dict(facecolor='white', edgecolor='none', alpha=0.8, pad=4))
    
    edges = G.edges(data=True)
    weights = [d['weight'] * 4 for u, v, d in edges] 
    
    nx.draw_networkx_edges(G, pos, width=weights, edge_color=edge_colors, 
                           arrowsize=35, arrowstyle='-|>', node_size=5500,        
                           connectionstyle="arc3,rad=0.15", min_source_margin=25,
                           min_target_margin=25, alpha=0.85)
    
    edge_labels = {(u, v): f"{d['weight']:.2f}" for u, v, d in G.edges(data=True)}
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=10, label_pos=0.7, rotate=False)

    plt.title(f"Red Bayesiana de Dependencia (Umbral > {threshold})", fontsize=16, fontweight='bold', pad=20)
    plt.axis('off')
    plt.tight_layout()

    # Guardado
    os.makedirs('Graficas', exist_ok=True)
    plt.savefig('Graficas/red_bayesiana_dependencias.svg', dpi=300, bbox_inches='tight')
    plt.show()

    return pd.DataFrame(relaciones_lista).sort_values(by='Probabilidad', ascending=False)

# EJECUCIÓN
dep_matrix = bayesian_network_of_dependencies(neo_driver)

if dep_matrix is not None:
    tabla_final = plot_bayesian_with_table_colored_save(dep_matrix, threshold=0.70)
    
    if tabla_final is not None and not tabla_final.empty:
        print("\n📋 DETALLE DE DEPENDENCIAS (REPORTE DE PROBABILIDAD CONDICIONAL):")
        
        # 1. Preparación de datos (Versión visual)
        df_itables = tabla_final.head(15).copy()
        df_itables['Probabilidad P(B|A)'] = (df_itables['Probabilidad'] * 100).map('{:,.2f}%'.format)
        df_itables = df_itables[['Causa (A)', 'Efecto (B)', 'Probabilidad P(B|A)']]

        display(HTML("""
        <style>
            .itables table { 
                margin-left: 0 !important; 
                margin-right: auto !important; 
                width: 700px !important; 
                max-width: 800px !important;
                font-size: 0.85em !important;
                border-collapse: collapse !important;
                table-layout: fixed !important; 
            }
            .itables th, .itables td { 
                padding: 10px 15px !important; 
                white-space: nowrap !important;
                overflow: hidden;
                text-overflow: ellipsis;
                text-align: left !important;
            }
            .itables th { background-color: #f7fafc !important; border-bottom: 2px solid #edf2f7 !important; }
            .itables td { border-bottom: 1px solid #edf2f7 !important; }
        </style>
        """))
        show(df_itables, 
             paging=False, 
             search=False, 
             info=False, 
             dom='t',
             autoWidth=False,
             columnDefs=[
                 {"width": "250px", "targets": 0}, # Causa (A)
                 {"width": "250px", "targets": 1}, # Efecto (B)
                 {"width": "200px", "targets": 2}, # Probabilidad %
                 {"className": "dt-left", "targets": "_all"}
             ])

In [ ]:
def bayesian_network_of_dependencies(neo_driver):
    print("🧮 Calculando probabilidades condicionales P(B|A)...")

    # CONSULTA
    query = """
    MATCH (p:Page)-[:MENTIONS]->(t:Term)
    RETURN p.url as doc_id, t.name as term_name
    UNION
    MATCH (p:Page)-[:MENTIONS]-(s:Synonym)-[:IS_SYNONYM_OF]->(t:Term)
    RETURN p.url as doc_id, t.name as term_name
    """
    
    try:
        with neo_driver.session() as session:
            result = session.run(query)
            datos = [dict(r) for r in result]
            
            if not datos: return None
            df_raw = pd.DataFrame(datos)
            if 'doc_id' not in df_raw.columns: return None

        presencia = pd.crosstab(df_raw['doc_id'], df_raw['term_name'])
        presencia = (presencia > 0).astype(int)
        
        nombres_terminos = presencia.columns
        matrix_data = []

        for term_a in nombres_terminos:
            fila = []
            count_a = presencia[term_a].sum()
            for term_b in nombres_terminos:
                if term_a == term_b:
                    fila.append(0.0)
                else:
                    ambos = ((presencia[term_a] == 1) & (presencia[term_b] == 1)).sum()
                    prob = ambos / count_a if count_a > 0 else 0
                    fila.append(prob)
            matrix_data.append(fila)
        
        return pd.DataFrame(matrix_data, index=nombres_terminos, columns=nombres_terminos)

    except Exception as e:
        print(f"❌ Error: {e}")
        return None

def plot_bayesian_with_table_colored_save(dep_matrix, threshold=0.70):
    if dep_matrix is None or dep_matrix.empty:
        return None
        
    G = nx.DiGraph() 
    relaciones_lista = []

    for term_a in dep_matrix.index:
        for term_b in dep_matrix.columns:
            prob = dep_matrix.loc[term_a, term_b]
            if prob >= threshold:
                G.add_edge(term_a, term_b, weight=prob)
                relaciones_lista.append({
                    'Causa (A)': term_a.upper(),
                    'Efecto (B)': term_b.upper(),
                    'Probabilidad': float(prob)
                })

    if not relaciones_lista:
        print(f"ℹ️ Ninguna relación supera el {threshold*100}%.")
        return pd.DataFrame()

    # =========================================================================
    # CONFIGURACIÓN DE RED DE ALTA VISIBILIDAD (ESPECIAL POWERPOINT/PDF)
    # =========================================================================
    plt.figure(figsize=(16, 13), facecolor='white') # Aumentado el lienzo horizontal y vertical
    pos = nx.circular_layout(G)
    
    node_colors = [colores_tfg.get(node.lower(), '#999') for node in G.nodes()]
    edge_colors = [colores_tfg.get(u.lower(), 'black') for u, v in G.edges()]
    
    # Nodos masivos y limpios para soportar textos grandes
    nx.draw_networkx_nodes(G, pos, node_size=6500, node_color=node_colors, 
                           edgecolors='black', linewidths=2.5, alpha=1.0)
    
    # Etiquetas de los nodos (¡Aumentadas a 13pt y en negrita!)
    nx.draw_networkx_labels(G, pos, font_size=13, font_weight='bold',
                            bbox=dict(facecolor='white', edgecolor='none', alpha=0.85, pad=5))
    
    edges = G.edges(data=True)
    weights = [d['weight'] * 4.5 for u, v, d in edges] # Un pelín más gruesas las líneas
    
    # Ajuste crítico de márgenes para que las flechas grandes no se escondan bajo los nodos
    nx.draw_networkx_edges(G, pos, width=weights, edge_color=edge_colors, 
                           arrowsize=45, arrowstyle='-|>', node_size=6500,        
                           connectionstyle="arc3,rad=0.18", min_source_margin=35,
                           min_target_margin=35, alpha=0.85)
    
    # Etiquetas de probabilidad sobre las líneas de unión (¡Subidas a 12pt!)
    edge_labels = {(u, v): f"{d['weight']:.2f}" for u, v, d in G.edges(data=True)}
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=12, 
                                 font_weight='bold', label_pos=0.68, rotate=False)

    # TÍTULO DESTACADO
    plt.title(f"Red Bayesiana de Dependencia (Umbral > {threshold})", fontsize=18, fontweight='bold', pad=30)
    plt.axis('off')
    plt.tight_layout()

    # GUARDADO VECTORIAL PREMIUM
    os.makedirs('Graficas', exist_ok=True)
    plt.savefig('Graficas/red_bayesiana_dependencias.svg', bbox_inches='tight')
    plt.show()

    return pd.DataFrame(relaciones_lista).sort_values(by='Probabilidad', ascending=False)

# EJECUCIÓN
dep_matrix = bayesian_network_of_dependencies(neo_driver)

if dep_matrix is not None:
    tabla_final = plot_bayesian_with_table_colored_save(dep_matrix, threshold=0.70)
    
    if tabla_final is not None and not tabla_final.empty:
        print("\n📋 DETALLE DE DEPENDENCIAS (REPORTE DE PROBABILIDAD CONDICIONAL):")
        
        # 1. Preparación de datos (Versión visual)
        df_itables = tabla_final.head(15).copy()
        df_itables['Probabilidad P(B|A)'] = (df_itables['Probabilidad'] * 100).map('{:,.2f}%'.format)
        df_itables = df_itables[['Causa (A)', 'Efecto (B)', 'Probabilidad P(B|A)']]

        display(HTML("""
        <style>
            .itables table { 
                margin-left: 0 !important; 
                margin-right: auto !important; 
                width: 700px !important; 
                max-width: 800px !important;
                font-size: 0.85em !important;
                border-collapse: collapse !important;
                table-layout: fixed !important; 
            }
            .itables th, .itables td { 
                padding: 10px 15px !important; 
                white-space: nowrap !important;
                overflow: hidden;
                text-overflow: ellipsis;
                text-align: left !important;
            }
            .itables th { background-color: #f7fafc !important; border-bottom: 2px solid #edf2f7 !important; }
            .itables td { border-bottom: 1px solid #edf2f7 !important; }
        </style>
        """))
        show(df_itables, 
             paging=False, 
             search=False, 
             info=False, 
             dom='t',
             autoWidth=False,
             columnDefs=[
                 {"width": "250px", "targets": 0}, # Causa (A)
                 {"width": "250px", "targets": 1}, # Efecto (B)
                 {"width": "200px", "targets": 2}, # Probabilidad %
                 {"className": "dt-left", "targets": "_all"}
             ])

### Análisis de Mann-Witney

Demuestra si usar una ontologia (agrupar para la bñusqueda término raíz y sus sinonimos) es mejor que buscar términos sueltos.  
Para eso se realiza primero un análisis indivudual de cada térimino con sus sinónimos y, finalmente, se pone todo en forma grupal, para poder ver sobre el total los resultados.

In [ ]:
# Silenciar avisos y warnings
opt.warn_on_undocumented_option = False
warnings.filterwarnings('ignore', category=SyntaxWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)

def validation_comprehensive_ontology(neo_driver):
    print("🔍 Iniciando Validación Detallada: Término a Término + Análisis Global...")

    # Consulta Neo4J
    query = """
    MATCH (t:Term)
    OPTIONAL MATCH (s:Synonym)-[:IS_SYNONYM_OF]->(t)
    WITH t, s
    OPTIONAL MATCH (s)-[:MENTIONS]-(p_sin:Page)
    WITH t, s.name as sinonimo, count(distinct p_sin) as menciones_por_sinonimo
    ORDER BY menciones_por_sinonimo DESC
    
    RETURN t.name AS Termino_Raiz, 
           collect({nombre: sinonimo, cuenta: menciones_por_sinonimo}) as detalle_sinonimos
    """
    
    try:
        with neo_driver.session() as session:
            res = session.run(query).data()
            
        if not res:
            print("❌ No hay datos en Neo4j para realizar la validación.")
            return

        impactos_agrupados = []
        impactos_individuales = []

        # Creación de la tabla
        estilo_tabla_comun = """
        <style>
            .itables table { 
                margin-left: 0 !important; margin-right: auto !important; 
                width: 700px !important; max-width: 700px !important;
                font-size: 0.85em !important; border-collapse: collapse !important;
            }
            .itables th { background-color: #f7fafc !important; color: #4a5568 !important; padding: 10px 15px !important; border-bottom: 2px solid #edf2f7 !important; text-align: left !important; }
            .itables td { padding: 8px 15px !important; color: #2d3748 !important; border-bottom: 1px solid #edf2f7 !important; }
            .total-row { font-weight: bold !important; background-color: #f0fff4 !important; color: #2f855a !important; }
        </style>
        """

        # FASE 1: ANÁLISIS INDIVIDUALIZADO
        print("\n" + "="*60)
        print("FASE 1: ANÁLISIS INDIVIDUAL (POR CATEGORÍA)")
        print("="*60 + "\n")
        
        for registro in res:
            termino = registro['Termino_Raiz']
            sinonimos = registro['detalle_sinonimos']
            
            datos_plot = [s for s in sinonimos if s['nombre'] is not None and s['cuenta'] > 0]
            if not datos_plot: continue

            nombres = [s['nombre'] for s in datos_plot]
            cuentas = [s['cuenta'] for s in datos_plot]
            total_termino = sum(cuentas)
            
            impactos_agrupados.append(total_termino)
            impactos_individuales.extend(cuentas)

            # Gráficas
            plt.figure(figsize=(9, 4))
            sns.barplot(x=nombres, y=cuentas, hue=nombres, palette='rocket', legend=False, edgecolor='black', alpha=0.7)
            plt.axhline(y=total_termino, color='red', linestyle='--', linewidth=2, label=f'Impacto Agrupado: {total_termino}')
            plt.title(f"Análisis Micro: {termino.upper()}", fontsize=13, fontweight='bold')
            plt.ylabel("Nº de Páginas")
            plt.xticks(rotation=25, ha='right')
            plt.legend()
            plt.grid(axis='y', linestyle='--', alpha=0.3)
            plt.tight_layout()
            plt.show()

            # Tablas
            df_micro = pd.DataFrame(datos_plot).rename(columns={'nombre': 'Sinónimo / Término', 'cuenta': 'Páginas'})
            fila_total = pd.DataFrame([{'Sinónimo / Término': 'TOTAL (ONTOLOGÍA)', 'Páginas': total_termino}])
            df_micro = pd.concat([df_micro, fila_total], ignore_index=True)
            
            print(f"📊 Evidencias para {termino.upper()}:")
            display(HTML(estilo_tabla_comun))
            show(df_micro, paging=False, search=False, info=False, dom='t',
                 columnDefs=[{"className": "dt-left", "targets": "_all"}])
            print("\n" + "—"*60 + "\n")

        # FASE 2: COMPARACIÓN FINAL GENERAL
        print("\n" + "="*60)
        print("FASE 2: COMPARACIÓN ESTADÍSTICA MACRO")
        print("="*60 + "\n")
        
        if not impactos_agrupados: return

        grupo_a = np.array(impactos_agrupados)
        grupo_b = np.array(impactos_individuales)

        u_stat, p_val = mannwhitneyu(grupo_a, grupo_b, alternative='greater')

        # Gráfica
        plt.figure(figsize=(10, 6))
        df_boxplot = pd.concat([
            pd.DataFrame({'Valor': grupo_a, 'Tipo': 'Modelo Ontológico (Suma)'}),
            pd.DataFrame({'Valor': grupo_b, 'Tipo': 'Términos Sueltos'})
        ])
        
        colores_macro = {
            'Modelo Ontológico (Suma)': '#38A169', 
            'Términos Sueltos': '#E53E3E'           
        }
        sns.boxplot(
            x='Tipo', y='Valor', data=df_boxplot, hue='Tipo', 
            palette=colores_macro, showfliers=False, legend=False,
            width=0.4
        )
        sns.stripplot(x='Tipo', y='Valor', data=df_boxplot, color=".2", size=6, alpha=0.4)
        
        plt.yscale('log')
        
        # Texto
        plt.title(f"Validación Final del Modelo\n(Mann-Whitney U | p-valor: {p_val:.6f})", fontsize=16, fontweight='bold', pad=15)
        plt.ylabel("Volumen de Evidencias (Log)", fontsize=13, fontweight='bold')
        plt.xlabel("Tipo", fontsize=13, fontweight='bold')
        
        plt.xticks(fontsize=13, fontweight='bold')
        plt.yticks(fontsize=11)
        
        plt.grid(axis='y', alpha=0.2)
        plt.tight_layout()
        
        # Guardado 
        os.makedirs('Graficas', exist_ok=True)
        plt.savefig('Graficas/validacion_macro_ontologia.svg', dpi=300, bbox_inches='tight', facecolor='white')
        plt.show()

        # Tabla
        resumen_global = pd.DataFrame({
            'Métrica': ['Suma Total de Evidencias', 'Promedio por Categoría', 'Nº de Elementos (n)'],
            'Modelo Agrupado': [f"{int(grupo_a.sum()):,}", f"{grupo_a.mean():.2f}", len(grupo_a)],
            'Términos Sueltos': [f"{int(grupo_b.sum()):,}", f"{grupo_b.mean():.2f}", len(grupo_b)]
        })
        
        print("📋 RESUMEN ESTADÍSTICO GLOBAL:")
        display(HTML(estilo_tabla_comun))
        show(resumen_global, paging=False, search=False, info=False, dom='t',
             columnDefs=[{"width": "250px", "targets": 0}, {"className": "dt-left", "targets": "_all"}])

        # Conclusión
        es_significativo = p_val < 0.05
        color_res = "#16a34a" if es_significativo else "#dc2626"
        display(HTML(f"""
            <div style="border: 2px solid {color_res}; padding:20px; border-radius:15px; background:#f8fafc; margin-top:20px; font-family: sans-serif; width: 660px;">
                <h3 style="color:{color_res}; margin:0;">Resultado: {"ESTADÍSTICAMENTE SUPERIOR ✅" if es_significativo else "SIN DIFERENCIA SIGNIFICATIVA ❌"}</h3>
                <p style="font-size: 1.1em; color: #334155;">
                    P-valor: <b>{p_val:.6f}</b>. <br>
                    Este análisis confirms que la arquitectura de la ontología incrementa significativamente 
                    la capacidad de recolección frente a términos aislados.
                </p>
            </div>
        """))

    except Exception as e:
        print(f"❌ Error en la ejecución: {e}")

# Ejecutar la validación
validation_comprehensive_ontology(neo_driver)

Con este código se puede ver si la diferenciación que se hizo de manera inicial por términos y sus sinonimos sería la correcta o si algún término es similar.

In [ ]:
itables.options.warn_on_undocumented_option = False

def mannwhitney_complete_analysis_save(neo_driver):
    print("🧪 Iniciando Análisis Completo Mann-Whitney U (Grafo + Tabla)...")
    
    try:
        # CONSULTA
        query_all = """
        MATCH (p:Page) WHERE p.text IS NOT NULL
        WITH p LIMIT 5000 
        MATCH (t:Term)
        OPTIONAL MATCH (p)-[:MENTIONS]->(s:Synonym)-[:IS_SYNONYM_OF]->(t)
        RETURN p.url as url, t.name as categoria, count(s) as intensidad
        """
        
        with neo_driver.session() as session:
            result = session.run(query_all)
            records = [dict(r) for r in result]
            
        if not records: return None

        df_long = pd.DataFrame(records)
        df = df_long.pivot(index='url', columns='categoria', values='intensidad').fillna(0)
        categorias = df.columns
        
        # GRAFICA Y TABLA
        G = nx.Graph() 
        for cat in categorias: G.add_node(cat.upper())
            
        resultados_lista = []
        
        print(f"📊 Analizando {len(categorias)} mercados...")
        
        for cat1, cat2 in combinations(categorias, 2):
            stat, p = mannwhitneyu(df[cat1], df[cat2], alternative='two-sided')

            if p > 0.05: 
                G.add_edge(cat1.upper(), cat2.upper(), weight=p)
                conclusio = "Similares (Mismo Patrón)"
                icono = "⚠️"
                nota = "Posible solapamiento"
            else:
                conclusio = "Diferentes (Distintos)"
                icono = "✅"
                nota = "Bien diferenciados"

            resultados_lista.append({
                'Mercado A': cat1.upper(),
                'Mercado B': cat2.upper(),
                'P-Valor': round(float(p), 4),
                'Resultado': f"{icono} {conclusio}",
                'Interpretación': nota
            })

        # GRAFICA
        plt.figure(figsize=(14, 10))
        pos = nx.spring_layout(G, k=0.5, iterations=50, seed=42)
        node_colors = [colores_tfg.get(node.lower(), '#999') for node in G.nodes()]
        
        nx.draw_networkx_nodes(G, pos, node_size=3500, node_color=node_colors, edgecolors='black', linewidths=1.5)
        nx.draw_networkx_labels(G, pos, font_size=9, font_weight='bold', font_color='white')
        
        edges = G.edges(data=True)
        if edges:
            weights = [d['weight'] * 5 for u, v, d in edges]
            nx.draw_networkx_edges(G, pos, width=weights, edge_color='#4A5568', style='dashed', alpha=0.6)
            edge_labels = {(u, v): f"p={d['weight']:.2f}" for u, v, d in edges}
            nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=8)

        ax = plt.gca()
        ax.margins(0.15)
        
        plt.title("Mapa de Similitud Estadística (Mann-Whitney U)\nNodos conectados = Mercados con comportamiento indistinguible", 
                  fontsize=16, fontweight='bold', pad=20)
        plt.axis('off')
        
        # GUARDADO
        os.makedirs('Graficas', exist_ok=True)
        plt.savefig('Graficas/grafo_similitud_mann_whitney.svg', dpi=300, bbox_inches='tight')
        plt.show()

        # TABLA 
        df_resumen = pd.DataFrame(resultados_lista).sort_values(by='P-Valor', ascending=False)

        print("\n--- 📋 DETALLE ESTADÍSTICO COMPLETO (ORDENADO POR SIMILITUD) ---")
        
        display(HTML("""
        <style>
            .itables table { 
                margin-left: 0 !important; 
                margin-right: auto !important; 
                width: 700px !important; 
                max-width: 800px !important;
                font-size: 0.85em !important;
                border-collapse: collapse !important;
            }
            .itables th { background-color: #f7fafc !important; color: #4a5568 !important; padding: 10px 15px !important; border-bottom: 2px solid #edf2f7 !important; text-align: left !important; }
            .itables td { padding: 8px 15px !important; color: #2d3748 !important; border-bottom: 1px solid #edf2f7 !important; text-align: left !important; }
        </style>
        """))

        show(df_resumen, 
             paging=False, 
             search=False, 
             info=False, 
             dom='t',
             columnDefs=[
                 {"width": "120px", "targets": 0}, # Mercado A
                 {"width": "120px", "targets": 1}, # Mercado B
                 {"width": "80px", "targets": 2},  # P-Valor
                 {"width": "180px", "targets": 3}, # Resultado
                 {"width": "200px", "targets": 4}, # Interpretación
                 {"className": "dt-left", "targets": "_all"}
             ])
        
        os.makedirs('Datos', exist_ok=True)
        df_resumen.to_csv('Datos/comparativa_mann_whitney_completa.csv', index=False)
        print(f"\n✅ Datos guardados en: Datos/comparativa_mann_whitney_completa.csv")

    except Exception as e:
        print(f"❌ Error: {e}")

# EJECUCIÓN
mannwhitney_complete_analysis_save(neo_driver)

<div class="seccion-ocultar">

### Análisis código profundidad 3

Al realizarse la ingesta, se pudo ver como en las primeras profundidades, todas las páginas correspondían a los términos por los cuales se filtraba la ingesta (los del diccionario creadoo al inicio) pero, al llegar a la profundidad 3, ya no se podían controlar los resultados que nos ofrecía el crawler, por lo tanto, se realiza este código para poder así ver que páginas pueden verse en esta profundidad si no controlamos lo que sale y, para así tambien poder ver que si se entra en la Dark Web es muy facil toparse con páginas cuyos contenidos son sensibles y no del interes del trabajo.

</div>

In [ ]:
def analizar_solo_limbo_depth3_con_totales_limpios(neo_driver, max_paginas=6000000):
    if not neo_driver:
        print("❌ Error: El driver de Neo4j no está disponible.")
        return None

    print(f"🌐 Analizando {max_paginas} páginas en profundidad 3 (Contador de Secciones Individuales)...")
    # consulta
    query = """
    MATCH (root:Page)
    WHERE NOT ()-[:LINKS_TO]->(root)
    
    MATCH (root)-[:LINKS_TO*3]->(p:Page)
    WITH DISTINCT p LIMIT $limit
    
    OPTIONAL MATCH (p)-[:MENTIONS]->(s:Synonym)-[:IS_SYNONYM_OF]->(t:Term)
    RETURN p.url AS url, p.title AS titulo, collect(DISTINCT t.name) AS terminos_oficiales
    """
    
    try:
        with neo_driver.session() as session:
            records = session.run(query, limit=max_paginas).data()
            
            if not records:
                print("⚠️ No se encontraron páginas en la profundidad 3.")
                return None
            
            df_raw = pd.DataFrame(records)

        df_raw['titulo'] = df_raw['titulo'].fillna('')
        df_raw['url'] = df_raw['url'].fillna('')

        datos_limbo = []
        todas_las_categorias_encontradas = []  

        for _, row in df_raw.iterrows():
            terminos = row['terminos_oficiales']
            
            # Si ya tiene terminos en el grafo seguimos
            if terminos and any(terminos):
                continue
                
            # Realiza una evaluación del titulo y del URL para poder tener más información
            titulo_raw = str(row['titulo']).strip()
            url_raw = str(row['url']).strip()
            texto_analisis = f"{titulo_raw.lower()} {url_raw.lower()}".strip()
            # Si no tiene titulo se guarda para poder tener más información
            tiene_titulo_vacio = (titulo_raw == '' or 'sin título' in titulo_raw.lower() or 'no title' in titulo_raw.lower())
            
            categorias_activadas = []
            
            # filtramos por lo que se sabe que hay en la profundidad para que lo clasifique
            if 'child' in texto_analisis or 'boy' in texto_analisis or 'girl' in texto_analisis or 'baby' in texto_analisis or 'toodler' in texto_analisis or 'children' in texto_analisis or 'yr' in texto_analisis or 'yo' in texto_analisis or 'years' in texto_analisis or 'year' in texto_analisis or 'old' in texto_analisis or 'kid' in texto_analisis or 'teen' in texto_analisis or 'babko' in texto_analisis or 'young' in texto_analisis or 'ped' in texto_analisis or 'daddy' in texto_analisis or 'young' in texto_analisis or 'father' in texto_analisis or 'daughter' in texto_analisis:
                categorias_activadas.append("Child P")
            
            if 'porn' in texto_analisis or 'sex' in texto_analisis or 'gay' in texto_analisis or 'lesbian' in texto_analisis or 'man' in texto_analisis or 'woman' in texto_analisis or 'video' in texto_analisis or 'dock' in texto_analisis or 'dick' in texto_analisis or 'puss' in texto_analisis or 'brutal' in texto_analisis or 'whore' in texto_analisis or 'clip' in texto_analisis or 'hentai' in texto_analisis or 'monica' in texto_analisis or 'galleries' in texto_analisis or 'nudist' in texto_analisis or 'xx' in texto_analisis or 'rape' in texto_analisis or 'loli' in texto_analisis or 'old' in texto_analisis or 'fuck' in texto_analisis or 'incest' in texto_analisis or 'milf' in texto_analisis or 'video' in texto_analisis or 'lip' in texto_analisis or 'feti' in texto_analisis:
                categorias_activadas.append("P")
            
            if 'animal' in texto_analisis or 'animals' in texto_analisis or 'horse' in texto_analisis or 'dog' in texto_analisis:
                categorias_activadas.append("Animal P")

            if 'organ' in texto_analisis or 'body' in texto_analisis:
                categorias_activadas.append("Organs")

            if 'dead' in texto_analisis or 'shot' in texto_analisis or 'murder' in texto_analisis or 'body' in texto_analisis:
                categorias_activadas.append("Murder")

            if 'tampa' in texto_analisis or 'customs' in texto_analisis or 'bay' in texto_analisis:
                categorias_activadas.append("False documents for customs")

            if 'border' in texto_analisis or 'camp' in texto_analisis:
                categorias_activadas.append("Human Smuggling")

            if 'money' in texto_analisis or 'make' in texto_analisis or 'fast' in texto_analisis or 'quickly' in texto_analisis:
                categorias_activadas.append("Fast Money")

            if len(categorias_activadas) > 0:
                categoria_final_texto = " y ".join(categorias_activadas)
                todas_las_categorias_encontradas.extend(categorias_activadas)
            elif tiene_titulo_vacio:
                categoria_final_texto = "Sin datos en título/URL"
                todas_las_categorias_encontradas.append("Sin datos en título/URL")
            else:
                categoria_final_texto = "Otras temáticas / Títulos específicos"
                todas_las_categorias_encontradas.append("Otras temáticas / Títulos específicos")

            titulo_visto = titulo_raw if titulo_raw != '' else '⚠️ [Sin título]'
            
            datos_limbo.append({
                "Título Real": titulo_visto,
                "Temática Deducida (Fuera de Diccionario)": categoria_final_texto,
                "URL": url_raw
            })

        if not datos_limbo:
            display(HTML("<div style='font-family: sans-serif; color: #2f855a;'>✨ <b>¡Perfecto!</b> Todo catalogado.</div>"))
            return None

        df_limbo = pd.DataFrame(datos_limbo)

        df_resumen = pd.Series(todas_las_categorias_encontradas).value_counts().reset_index()
        df_resumen.columns = ["Sección / Categoría del Limbo", "Total de Impactos Detectados"]
        display(HTML(f"""
        <div style="font-family: sans-serif; max-width: 1000px; margin-top: 20px;">
            <h2 style="color: #c53030; border-bottom: 3px solid #c53030; padding-bottom: 8px;">🛑 Métricas por Secciones Limpias (Profundidad 3)</h2>
            <p style="color: #4a5568; font-size: 14px;">Este resumen cuenta cada sección por separado. Si una página pertenecía a dos temáticas a la vez, ha sumado +1 en cada una de sus respectivas filas.</p>
            
            <h4 style="color: #2d3748; margin-top: 20px; margin-bottom: 5px;">📊 Total Neto por Categoría</h4>
        </div>
        """))
        
        display(HTML(f"<div style='max-width: 650px; margin-bottom: 25px;'>{df_resumen.to_html(index=False, classes='table table-striped')}</div>"))
        
        display(HTML(f"""
        <div style="font-family: sans-serif; max-width: 1000px;">
            <h4 style="color: #2d3748; margin-bottom: 5px;">🔍 Listado Detallado de Muestra (Sin Ruido)</h4>
        </div>
        """))
        
        df_detallado_visible = df_limbo[df_limbo["Temática Deducida (Fuera de Diccionario)"] != "Sin datos en título/URL"]
        
        if not df_detallado_visible.empty:
            df_tabla_visible = df_detallado_visible.head(20).copy()
            df_tabla_visible['URL'] = df_tabla_visible['URL'].apply(lambda u: f"<code style='font-size:10px;'>{u[:60]}...</code>")
            display(HTML(f"<div style='max-width: 1000px;'>{df_tabla_visible.to_html(escape=False, index=False, classes='table table-striped')}</div>"))
            
            paginas_restantes_visibles = len(df_detallado_visible) - len(df_tabla_visible)
            if paginas_restantes_visibles > 0:
                print(f"   ... y otras {paginas_restantes_visibles} páginas más registradas en el DataFrame.")
        else:
            display(HTML("<p style='color: #4a5568; font-style: italic; font-size: 13px;'>No hay páginas semánticas legibles.</p>"))
        
        return df_limbo

    except Exception as e:
        print(f"❌ Error al ejecutar el conteo limpio por secciones: {e}")
        return None

# ejecución
df_cosas_por_clasificar = analizar_solo_limbo_depth3_con_totales_limpios(neo_driver, max_paginas=6000000)

<div class="seccion-ocultar">

### Impresión del PDF final

</div>

<div class="seccion-ocultar">

Se imprime el PDF con todas las explicaciones (markdown) y las graficas con sus respectivas tablas.

</div>

In [ ]:
#OCULTAR
# CONFIGURACIÓN
nombre_notebook = "Analisis.ipynb" 
nombre_pdf_final = "TFG_Sandra_Final_Ajustado.pdf"

print("🎨 Generando el PDF filtrado...")

comando = [
    sys.executable, "-m", "nbconvert", 
    "--to", "webpdf",
    "--no-input", 
    "--TagRemovePreprocessor.remove_cell_tags={'ocultar'}", 
    "--WebPDFExporter.metadata={'orientation': 'landscape'}", 
    "--WebPDFExporter.scale=0.6", 
    "--WebPDFExporter.margin=5mm",
    "--allow-chromium-download",
    nombre_notebook,
    "--output", nombre_pdf_final
]

try:
    resultado = subprocess.run(comando, capture_output=True, text=True, check=True)
    print(f"✅ ¡CONSEGUIDO! PDF generado sin los elementos ocultos.")
    print(f"📂 Ubicación: {os.getcwd()}/{nombre_pdf_final}")
except subprocess.CalledProcessError as e:
    print("❌ ERROR CRÍTICO:")
    print(e.stderr)

<div class="seccion-ocultar">

### Botón para ocultar códigos

</div>

In [ ]:
def boton_ocultar_codigos():
    html_code = """
    <script>
    (function() {
        // 1. Por si ya ha sido ejecutado, limpiarlo
        const viejoPanel = document.getElementById('tfg-panel-global');
        if (viejoPanel) viejoPanel.remove();

        // 2. Se crea el botón
        const panel = document.createElement('div');
        panel.id = 'tfg-panel-global';
        panel.style.cssText = 'position: fixed; bottom: 20px; right: 20px; z-index: 9999999;';
        
        const btn = document.createElement('button');
        btn.id = 'btn-toggle-tfg-global';
        btn.innerHTML = "Vista Informe (Ocultar Código)";
        btn.style.cssText = `
            background-color: #2b6cb0; color: white; padding: 12px 24px; 
            border: none; border-radius: 50px; font-weight: bold; 
            cursor: pointer; box-shadow: 0 4px 15px rgba(0,0,0,0.4);
            font-family: sans-serif; transition: all 0.3s;
        `;

        panel.appendChild(btn);
        document.body.appendChild(panel);

        // 3. Normas que sigue el código para ocultar
        window.intercambiarVistaTFG = function() {
            let estilo = document.getElementById('estilo-tfg-global');
            
            if (!estilo) {
                // Crear el estilo que oculta las celdas
                estilo = document.createElement('style');
                estilo.id = 'estilo-tfg-global';
                estilo.innerHTML = `
                    /* Oculta la entrada (código) de todas las celdas */
                    .jp-CodeCell .jp-Cell-inputWrapper, 
                    .cell-code-area, 
                    .input_area { 
                        display: none !important; 
                    }
                    /* Oculta las salidas que tengan la clase específica */
                    .tfg-oculto-js, .seccion-ocultar { 
                        display: none !important; 
                    }
                `;
                document.head.appendChild(estilo);

                // Buscar celdas con el tag #OCULTAR para ocultar sus salidas también
                document.querySelectorAll('.jp-Cell, .cell').forEach(celda => {
                    if (celda.textContent.includes('#OCULTAR')) {
                        const output = celda.querySelector('.jp-Cell-outputWrapper, .output_wrapper');
                        if (output) output.classList.add('tfg-oculto-js');
                    }
                });

                btn.innerHTML = "Modo Edición (Mostrar Código)";
                btn.style.backgroundColor = "#4a5568";
            } else {
                // Volver al modo edición
                estilo.remove();
                document.querySelectorAll('.tfg-oculto-js').forEach(el => el.classList.remove('tfg-oculto-js'));
                btn.innerHTML = "Vista Informe (Ocultar Código)";
                btn.style.backgroundColor = "#2b6cb0";
            }
        };

        btn.onclick = window.intercambiarVistaTFG;
    })();
    </script>
    """
    display(HTML(html_code))

boton_ocultar_codigos()